### Configurations

In [ ]:
%%capture
# no Colab instala LaTeX e cmcrameri; no PC instale antes (ver README)
import os, subprocess, sys
if os.path.exists("/content"):
    subprocess.run("apt-get update -qq && apt-get install -y -qq texlive texlive-latex-extra texlive-fonts-recommended dvipng cm-super", shell=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cmcrameri"])


In [ ]:
import os
import glob
import time
import uuid
import copy
import random
import itertools
import colorsys

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as colors
import matplotlib.patheffects as pe
import matplotlib.ticker as ticker

from itertools import permutations

from scipy import integrate
from scipy.integrate import quad
from scipy.signal import find_peaks, peak_widths
from scipy.special import (
    eval_legendre,
    lpmv,
    spherical_jn,
    spherical_yn,
)

from matplotlib.colors import (
    LogNorm,
    LinearSegmentedColormap,
    BoundaryNorm,
    ListedColormap,
)

from matplotlib.ticker import (
    ScalarFormatter,
    LogLocator,
    LogFormatter,
    NullFormatter,
)

from matplotlib.patches import Rectangle
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.axes_grid1 import make_axes_locatable

try:
    import cmcrameri.cm as cmc
except Exception:
    cmc = None


plt.rcParams.update({
    "text.usetex"   : True,
    "figure.dpi"    : 600,
    "font.family" : "sans-serif",
})

### Integrals

\begin{equation}
\begin{split}
F_{jj}(x_B,y_B)
&=
\left[
x^3
\left(
j_\ell^2(x)
-
j_{\ell-1}(x)j_{\ell+1}(x)
\right)
\right]_{x_B}^{y_B},
\\
F_{yy}(x_B,y_B)
&=
\left[
x^3
\left(
y_\ell^2(x)
-
y_{\ell-1}(x)y_{\ell+1}(x)
\right)
\right]_{x_B}^{y_B},
\\
F_{jy}(x_B,y_B)
&=
\left[
\frac{x^3}{2}
\left(
2j_\ell(x)y_\ell(x)
-
j_{\ell+1}(x)y_{\ell-1}(x)
-
j_{\ell-1}(x)y_{\ell+1}(x)
\right)
\right]_{x_B}^{y_B},
\end{split}
\end{equation}

\begin{equation}
\begin{split}
B_{jj}(x_B,y_B)
&=
\left[
x^2
\left(
j_\ell(x)*j'_\ell(x)
\right)
\right]_{x_B}^{y_B},
\\
B_{yy}(x_B,y_B)
&=
\left[
x^2
\left(
y_\ell(x)*y'_\ell(x)
\right)
\right]_{x_B}^{y_B},
\\
B_{jy}(x_B,y_B)
&=
\left[
x^2
\left(
j_\ell(x)*y'_\ell(x)
\right)
\right]_{x_B}^{y_B},
\end{split}
\end{equation}

In [ ]:
def Fjj(l, xB, yB):
    def F(x):
        return x**3 * (J(l, x)**2 - J(l - 1, x) * J(l + 1, x))
    return F(yB) - F(xB)


def Fyy(l, xB, yB):
    def F(x):
        return x**3 * (Y(l, x)**2 - Y(l - 1, x) * Y(l + 1, x))
    return F(yB) - F(xB)


def Fjy(l, xB, yB):
    def F(x):
        return 0.5 * x**3 * (2 * J(l, x) * Y(l, x)- J(l + 1, x) * Y(l - 1, x)- J(l - 1, x) * Y(l + 1, x))
    return F(yB) - F(xB)


def Bjj(l, xB, yB):
    def B(x):
        return x**2 * J(l, x) * dJ(l, x)

    return B(yB) - B(xB)


def Byy(l, xB, yB):
    def B(x):
        return x**2 * Y(l, x) * dY(l, x)
    return B(yB) - B(xB)


def Bjy(l, xB, yB):
    def B(x):
        return x**2 * J(l, x) * dY(l, x)
    return B(yB) - B(xB)


def FA(l, xA):
    return 0.5 * xA**3 * (J(l, xA)**2 - J(l - 1, xA) * J(l + 1, xA))


def BA(l, xA):
    return xA**2 * J(l, xA) * dJ(l, xA)


def ell_max(x0_max):
    x0_max = float(x0_max)
    return int(3 + x0_max + 4.05 * x0_max**(1/3))


def get_lmax_core_shell(xM_vals, delta, c0, c1, c2, pad_core=0, pad_shell=0):
    xM_vals = np.asarray(xM_vals, dtype=float)
    xM_max = np.max(xM_vals)

    xA_max = (c0 / c1) * xM_max
    xB_max = (c0 / c2) * xM_max
    yB_max = (1.0 + delta) * xB_max

    lmax_core = ell_max(xA_max) + int(pad_core)
    lmax_shell = ell_max(yB_max) + int(pad_shell)

    return lmax_core, lmax_shell

### Bessel spherical functions




In [ ]:
def J(n, x):
    if n >= 0:
        return spherical_jn(n, x)
    m = -n - 1
    return (-1)**(m+1) * spherical_yn(m, x)

def Y(n, x):
    if n >= 0:
        return spherical_yn(n, x)
    m = -n - 1
    return (-1)**m * spherical_jn(m, x)

def dJ(n, x):
    if n >= 0:
        return spherical_jn(n, x, derivative=True)
    m = -n - 1
    return (-1)**(m+1) * spherical_yn(m, x, derivative=True)

def dY(n, x):
    if n >= 0:
        return spherical_yn(n, x, derivative=True)
    m = -n - 1
    return (-1)**m * spherical_jn(m, x, derivative=True)

def H(n, x):
    if n >= 0:
        return spherical_jn(n, x) + 1j * spherical_yn(n, x)
    m = -n - 1
    return 1j * (-1)**m * (spherical_jn(m, x) + 1j * spherical_yn(m, x))

def dH(n, x):
    if n >= 0:
        return spherical_jn(n, x, derivative=True) + 1j * spherical_yn(n, x, derivative=True)
    m = -n - 1
    return 1j * (-1)**m * (
        spherical_jn(m, x, derivative=True) + 1j * spherical_yn(m, x, derivative=True)
    )

### Coefficients

In [ ]:
def D_l_coreshell(l, xA, xB, yM, yB, rho0, rho1, rho2, c0, c1, c2):
    rho_MB = rho0 / rho2
    rho_AB = rho1 / rho2
    k_MB = c2 / c0
    k_AB = c2 / c1

    A = -J(l, yB)
    B = -Y(l, yB)
    F = rho_MB * H(l, yM)

    G = -dJ(l, yB)
    HH = -dY(l, yB)
    JJ = k_MB * dH(l, yM)

    L = -rho_AB * J(l, xA)
    M = J(l, xB)
    P = Y(l, xB)

    Q = -k_AB * dJ(l, xA)
    R = dJ(l, xB)
    S = dY(l, xB)

    return (
        (L * R - M * Q) * (F * HH - B * JJ)
        + (L * S - P * Q) * (A * JJ - F * G)
    )

def coeffs(l, xA, xB, yM, yB, rho0, rho1, rho2, c0, c1, c2):
    rho_MB = rho0 / rho2
    rho_AB = rho1 / rho2

    k_MB = c2 / c0
    k_AB = c2 / c1

    A = -J(l, yB)
    B = -Y(l, yB)
    F = rho_MB * H(l, yM)

    G = -dJ(l, yB)
    HH = -dY(l, yB)
    JJ = k_MB * dH(l, yM)

    L = -rho_AB * J(l, xA)
    M = J(l, xB)
    P = Y(l, xB)

    Q = -k_AB * dJ(l, xA)
    R = dJ(l, xB)
    S = dY(l, xB)


    D = ((L*R - M*Q) * (F*HH - B*JJ)+ (L*S - P*Q) * (A*JJ - F*G))

    X = 1j * rho_MB * k_MB / (yM**2)

    b_l = (M*S - P*R) * X / D
    c_l = (P*Q - L*S) * X / D
    d_l = (L*R - M*Q) * X / D

    return b_l, c_l, d_l

### Core energy

For the core, the potential and kinetic energies are evaluated directly from the
closed radial expressions. With $x_A=k_Aa$ and $Z_A=\rho_Ac_A$,

\begin{equation}
\frac{2 f_0 W_P^{(A)}}{Z_A |A|^2}
=
\sum_{\ell=0}^{\infty}
|b_\ell|^2(2\ell+1)
\frac{x_A^3}{2}
\left[
j_\ell^2(x_A)
-
j_{\ell-1}(x_A)j_{\ell+1}(x_A)
\right],
\end{equation}

\begin{equation}
\frac{2 f_0 W_K^{(A)}}{Z_A |A|^2}
=
\sum_{\ell=0}^{\infty}
|b_\ell|^2(2\ell+1)
\left\{
\frac{x_A^3}{2}
\left[
j_\ell^2(x_A)
-
j_{\ell-1}(x_A)j_{\ell+1}(x_A)
\right]
+
x_A^2 j_\ell(x_A)j_\ell'(x_A)
\right\}.
\end{equation}

Thus, the total core energy is simply

\begin{equation}
W_T^{(A)} = W_P^{(A)} + W_K^{(A)} .
\end{equation}

This form avoids the numerical evaluation of the auxiliary radial integrals used
in the previous implementation.

In [ ]:
def W_core_parts_vs_xM(xM_vals, delta, rho0, rho1, rho2, c0, c1, c2, lmax_core=None, pad_core=0, lmax_cap=None):

    xM_vals = np.asarray(xM_vals, dtype=float)

    WK_core = np.zeros_like(xM_vals, dtype=float)
    WP_core = np.zeros_like(xM_vals, dtype=float)

    if lmax_core is None:
        lmax_core, _ = get_lmax_core_shell(
            xM_vals,
            delta,
            c0, c1, c2,
            pad_core=pad_core,
            pad_shell=0
        )

    if lmax_cap is not None:
        lmax_core = min(lmax_core, int(lmax_cap))

    for ia, xM in enumerate(xM_vals):

        xA = (c0 / c1) * xM
        xB = (c0 / c2) * xM
        yM = (1.0 + delta) * xM
        yB = (1.0 + delta) * xB

        WKc = 0.0
        WPc = 0.0

        for l in range(lmax_core + 1):

            bl, cl, dl = coeffs(
                l,
                xA, xB, yM, yB,
                rho0, rho1, rho2,
                c0, c1, c2
            )

            bl2 = np.abs(bl)**2

            FA_l = FA(l, xA)
            BA_l = BA(l, xA)

            WPc += bl2 * (2*l + 1) * FA_l
            WKc += bl2 * (2*l + 1) * (FA_l + BA_l)

        WP_core[ia] = np.real_if_close(WPc)
        WK_core[ia] = np.real_if_close(WKc)

    return WK_core, WP_core


def WT_core_vs_xM(xM_vals, delta, rho0, rho1, rho2, c0, c1, c2, lmax_core=None, pad_core=0, lmax_cap=None):

    WK_core, WP_core = W_core_parts_vs_xM(
        xM_vals,
        delta,
        rho0, rho1, rho2,
        c0, c1, c2,
        lmax_core=lmax_core,
        pad_core=pad_core,
        lmax_cap=lmax_cap)

    return WK_core + WP_core

### Shell energy

For the shell, the potential energy is written as

\begin{equation}
\frac{2f_0W_P^{(B)}}{Z_B|A|^2}
=
\sum_{\ell=0}^{\infty}
(2\ell+1)
\left[
\frac{|c_\ell|^2}{2}F_{jj}(x_B,y_B)
+
\frac{|d_\ell|^2}{2}F_{yy}(x_B,y_B)
+
\mathrm{Re}(c_\ell^*d_\ell)F_{jy}(x_B,y_B)
\right],
\end{equation}

where

\begin{equation}
F_{jj}(x_B,y_B)
=
\left[
x^3
\left(
j_\ell^2(x)
-
j_{\ell-1}(x)j_{\ell+1}(x)
\right)
\right]_{x_B}^{y_B},
\end{equation}

\begin{equation}
F_{yy}(x_B,y_B)
=
\left[
x^3
\left(
y_\ell^2(x)
-
y_{\ell-1}(x)y_{\ell+1}(x)
\right)
\right]_{x_B}^{y_B},
\end{equation}

and

\begin{equation}
F_{jy}(x_B,y_B)
=
\left[
\frac{x^3}{2}
\left(
2j_\ell(x)y_\ell(x)
-
j_{\ell+1}(x)y_{\ell-1}(x)
-
j_{\ell-1}(x)y_{\ell+1}(x)
\right)
\right]_{x_B}^{y_B}.
\end{equation}

Using the closed form obtained after the cancellation between the radial and
angular contributions, the kinetic energy can be evaluated directly as

\begin{equation}
\begin{split}
\frac{2f_0W_K^{(B)}}{Z_B|A|^2}
=
\frac{2f_0W_P^{(B)}}{Z_B|A|^2}
+
\sum_{\ell=0}^{\infty}
(2\ell+1)
\Bigg\{
&
|c_\ell|^2
\left[
x^2j_\ell(x)j_\ell'(x)
\right]_{x_B}^{y_B}
\\
&
+
|d_\ell|^2
\left[
x^2y_\ell(x)y_\ell'(x)
\right]_{x_B}^{y_B}
\\
&
+
2\mathrm{Re}(c_\ell^*d_\ell)
\left[
x^2j_\ell(x)y_\ell'(x)
\right]_{x_B}^{y_B}
\Bigg\}.
\end{split}
\end{equation}

Thus, the total energy stored in the shell is

\begin{equation}
W_T^{(B)} = W_P^{(B)} + W_K^{(B)}.
\end{equation}

This form avoids the explicit numerical evaluation of the auxiliary integrals
\(I_{j,\ell}\), \(I_{y,\ell}\), and \(I_{jy,\ell}\), since these terms cancel when
the radial and angular kinetic contributions are combined.

In [ ]:
def W_shell_parts_vs_xM(
    xM_vals,
    delta,
    rho0, rho1, rho2,
    c0, c1, c2,
    lmax_shell=None,
    pad_shell=0,
    lmax_cap=None):


    xM_vals = np.asarray(xM_vals, dtype=float)

    WK_shell = np.zeros_like(xM_vals, dtype=float)
    WP_shell = np.zeros_like(xM_vals, dtype=float)

    if lmax_shell is None:
        _, lmax_shell = get_lmax_core_shell(
            xM_vals,
            delta,
            c0, c1, c2,
            pad_core=0,
            pad_shell=pad_shell
        )

    if lmax_cap is not None:
        lmax_shell = min(lmax_shell, int(lmax_cap))

    for ia, xM in enumerate(xM_vals):

        xA = (c0 / c1) * xM
        xB = (c0 / c2) * xM
        yM = (1.0 + delta) * xM
        yB = (1.0 + delta) * xB

        WKs = 0.0
        WPs = 0.0

        for l in range(lmax_shell + 1):

            bl, cl, dl = coeffs(l, xA, xB, yM, yB, rho0, rho1, rho2, c0, c1, c2)

            cl2 = np.abs(cl)**2
            dl2 = np.abs(dl)**2
            Re_cd = np.real(np.conj(cl) * dl)

            termo_P = (0.5 * cl2 * Fjj(l, xB, yB) + 0.5 * dl2 * Fyy(l, xB, yB) + Re_cd * Fjy(l, xB, yB))

            termo_B = (cl2 * Bjj(l, xB, yB) + dl2 * Byy(l, xB, yB) + 2.0 * Re_cd * Bjy(l, xB, yB))

            WPs += (2*l + 1) * termo_P
            WKs += (2*l + 1) * (termo_P + termo_B)

        WP_shell[ia] = np.real_if_close(WPs)
        WK_shell[ia] = np.real_if_close(WKs)

    return WK_shell, WP_shell


def WT_shell_vs_xM(xM_vals, delta, rho0, rho1, rho2, c0, c1, c2, lmax_shell=None, pad_shell=0, lmax_cap=None):

    WK_shell, WP_shell = W_shell_parts_vs_xM(xM_vals, delta, rho0, rho1, rho2, c0, c1, c2, lmax_shell=lmax_shell, pad_shell=pad_shell, lmax_cap=lmax_cap)

    return WK_shell + WP_shell

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, NullFormatter, FuncFormatter
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from scipy.signal import find_peaks


# ============================================================
# Assume já implementados e importados:
#   coeffs, FA, BA, Fjj, Fyy, Fjy, Bjj, Byy, Bjy,
#   get_lmax_core_shell,
#   W_core_parts_vs_xM, WT_core_vs_xM,
#   W_shell_parts_vs_xM, WT_shell_vs_xM
# ============================================================

LMAX = 10


# ============================================================
# ESTÉTICA
# ============================================================
COLORS = ["#0f223d", "#810000", "black"]
LINESTYLES = ["-", "--", "-."]
FIGSIZE = (6.5, 5.0)

LABEL_FONTSIZE = 20
TICK_FONTSIZE = 16
LEGEND_FONTSIZE = 15

plt.rcParams.update({
    "font.size": TICK_FONTSIZE,
    "axes.labelsize": LABEL_FONTSIZE,
    "xtick.labelsize": TICK_FONTSIZE,
    "ytick.labelsize": TICK_FONTSIZE,
})


# ============================================================
# COEFICIENTES DA RELAÇÃO DE EXISTÊNCIA (contraste na velocidade, dataset coreshell_final/resultados_mix)
# ============================================================
COEF_G = dict(
    c0=-3.536863223462e+00,
    c1=-1.472216155442e+00,
    c2=-1.163130944691e+00,
    c3=4.246050955653e+00,
    c4=1.694216135501e-01,
    c5=-2.154035917812e-01,
    c6=-6.061917745350e-02,
    c7=2.777016064863e-01,
    c8=-3.581962971027e-01,
    c9=-1.779651600840e+00
)


# ============================================================
# RELAÇÃO DE EXISTÊNCIA
# ============================================================
def g_velocity(alpha, beta, delta):

    la = np.log(alpha)
    lb = np.log(beta)

    c = COEF_G

    return (
        c["c0"]
        + c["c3"] * delta
        + c["c9"] * delta**2
        + la * (
            c["c1"]
            + c["c6"] * delta
            + c["c4"] * la
            + c["c5"] * lb
        )
        + lb * (
            c["c2"]
            + c["c8"] * delta
            + c["c7"] * lb
        )
    )


def P_res(alpha, beta, delta):

    g = g_velocity(
        alpha,
        beta,
        delta
    )

    return 1.0 / (
        1.0 + np.exp(-g)
    )


# ============================================================
# FORMATAÇÃO NUMÉRICA: PADRÃO BRASILEIRO
# ============================================================
def decimal_mathtext(text):
    """
    Substitui o ponto decimal por vírgula em expressões
    matemáticas, utilizando {,} para evitar espaçamento
    inadequado no MathText.
    """

    return str(text).replace(".", "{,}")


def fmt_decimal(value, decimals=2):
    """
    Formata número decimal com vírgula para texto comum.
    """

    return f"{value:.{decimals}f}".replace(".", ",")


def fmt_decimal_math(value, decimals=2):
    """
    Formata número decimal com vírgula para uso dentro
    de expressões matemáticas.
    """

    txt = f"{value:.{decimals}f}"

    return decimal_mathtext(txt)


def fmt_latex(val):
    """
    Formata números para uso em expressões matemáticas,
    utilizando vírgula decimal e notação científica
    em potência de 10.
    """

    if val == 0:
        return "0"

    exp = int(
        np.floor(
            np.log10(
                abs(val)
            )
        )
    )

    # --------------------------------------------------------
    # Valores entre 10^-2 e 10^2
    # --------------------------------------------------------
    if -2 <= exp <= 2:

        txt = f"{val:.3g}"

        return decimal_mathtext(txt)

    # --------------------------------------------------------
    # Notação científica
    # --------------------------------------------------------
    mantissa = val / (10**exp)

    if np.isclose(
        mantissa,
        1.0
    ):
        return f"10^{{{exp}}}"

    if np.isclose(
        mantissa,
        -1.0
    ):
        return f"-10^{{{exp}}}"

    mantissa_txt = decimal_mathtext(
        f"{mantissa:.1f}"
    )

    return (
        f"{mantissa_txt}"
        f" \\times 10^{{{exp}}}"
    )


def format_x_tick(value, pos=None):
    """
    Formatação dos ticks do eixo horizontal com
    vírgula como separador decimal.
    """

    if np.isclose(value, round(value)):
        return f"{value:.0f}"

    return (
        f"{value:.2f}"
        .rstrip("0")
        .rstrip(".")
        .replace(".", ",")
    )


# ============================================================
# SISTEMAS FÍSICOS
#
# alpha = Z_A / Z_M
# beta  = Z_B / Z_M
# ============================================================
systems = {

    "Liu et al. 2000": dict(

        rho0=1.18e3,
        c0=2539.52,

        rho1=11.6e3,
        c1=2493.09,

        rho2=1.3e3,
        c2=22.8709,

        delta=0.4200,

        yM_known=None,

        fname="validation_liu2000",

        y_ticks=np.array([
            1e-7,
            1e-3,
            1e1,
            1e5,
            1e9
        ]),

        ylim=(
            1e-7,
            1e10
        ),
    ),


    "Marston 2025": dict(

        rho0=1.22,
        c0=343.0,

        rho1=1.22,
        c1=343.0,

        rho2=841.8 * 1.22,
        c2=4.519 * 343.0,

        delta=(1 - 0.7) / 0.7,

        yM_known=0.1554,

        fname="validation_marston2025",

        y_ticks=np.array([
            1e-15,
            1e-11,
            1e-7,
            1e-3,
            1e1
        ]),

        ylim=(
            1e-15,
            1e2
        ),
    ),
}


author_labels = {

    "Liu et al. 2000":
        "Liu et al. (2000)",

    "Marston 2025":
        "Marston (2025)",
}


# ============================================================
# CÁLCULO
# ============================================================
xM_vals = np.linspace(
    1e-4,
    0.75,
    1000
)

I0_over_A0sq = 1.0

results = {}


for name, p in systems.items():

    # --------------------------------------------------------
    # Propriedades dos materiais
    # --------------------------------------------------------
    rho0 = p["rho0"]
    rho1 = p["rho1"]
    rho2 = p["rho2"]

    c0 = p["c0"]
    c1 = p["c1"]
    c2 = p["c2"]

    delta = p["delta"]


    # --------------------------------------------------------
    # Contrastes de impedância
    # --------------------------------------------------------
    alpha = (
        rho1 * c1
    ) / (
        rho0 * c0
    )

    beta = (
        rho2 * c2
    ) / (
        rho0 * c0
    )


    # --------------------------------------------------------
    # Parâmetro eta
    # --------------------------------------------------------
    eta = (
        alpha - beta
    ) / (
        alpha + beta
    )


    # --------------------------------------------------------
    # Energia no núcleo
    # --------------------------------------------------------
    WT_core = WT_core_vs_xM(
        xM_vals,
        delta,
        rho0,
        rho1,
        rho2,
        c0,
        c1,
        c2,
        lmax_core=LMAX,
        lmax_cap=LMAX,
    )


    # --------------------------------------------------------
    # Energia na casca
    # --------------------------------------------------------
    WT_shell = WT_shell_vs_xM(
        xM_vals,
        delta,
        rho0,
        rho1,
        rho2,
        c0,
        c1,
        c2,
        lmax_shell=LMAX,
        lmax_cap=LMAX,
    )


    # --------------------------------------------------------
    # Impedâncias
    # --------------------------------------------------------
    Z_A = rho1 * c1
    Z_B = rho2 * c2


    # --------------------------------------------------------
    # Energia interna total
    # --------------------------------------------------------
    W_total = (
        2
        * I0_over_A0sq
        * (
            WT_core / Z_A
            + WT_shell / Z_B
        )
    )


    # ========================================================
    # Parâmetro de tamanho externo
    #
    # y_M = (1 + delta) x_M
    # ========================================================
    yM_vals = (
        1 + delta
    ) * xM_vals


    # --------------------------------------------------------
    # Pontos válidos
    # --------------------------------------------------------
    valid = (
        np.isfinite(W_total)
        & (W_total > 0)
    )


    # --------------------------------------------------------
    # Logaritmo da energia para detecção de picos
    # --------------------------------------------------------
    logW = np.full_like(
        W_total,
        -np.inf
    )

    logW[valid] = np.log10(
        W_total[valid]
    )


    # --------------------------------------------------------
    # Detecção de ressonâncias
    # --------------------------------------------------------
    idx, _ = find_peaks(
        logW,
        prominence=0.75,
    )


    peaks_yM = yM_vals[idx]
    peaks_W = W_total[idx]


    # --------------------------------------------------------
    # Armazenamento
    # --------------------------------------------------------
    results[name] = dict(

        alpha=alpha,
        beta=beta,
        eta=eta,
        delta=delta,

        yM_vals=yM_vals,

        W_total=W_total,

        peaks_yM=peaks_yM,
        peaks_W=peaks_W,

        P_res=P_res(
            alpha,
            beta,
            delta
        ),

        yM_known=p["yM_known"],

        fname=p["fname"],

        y_ticks=p["y_ticks"],

        ylim=p["ylim"],
    )


# ============================================================
# GRÁFICO
# ============================================================
# ============================================================
# REGIME DE CADA SISTEMA E LINHAS CINZA PELA RELAÇÃO DE RECORRÊNCIA
#   Liu: o contraste do núcleo vem da densidade e o da casca, da velocidade -> regime misto (f = 0,5)
#   Marston: meio e núcleo de ar, contraste dominado pela densidade -> regime de densidade (f = 1)
#   demais sistemas: regime de velocidade (f = 0)
#   P_res usa os coeficientes c0..c9 do regime (dataset coreshell_final/resultados_mix).
#   Linhas cinza: x_(n+1) = x_ref A(eta,delta) + B(eta,delta) x_n, x_ref = sqrt(3 alpha^(2-f)),
#   geradas a partir do primeiro pico resolvido; o regime de densidade não tem recorrência,
#   então nele a única linha é a do primeiro pico.
# ============================================================
COEF_G_REGIME = {
    "velocity": [-3.536863223462e+00, -1.472216155442e+00, -1.163130944691e+00, 4.246050955653e+00, 1.694216135501e-01, -2.154035917812e-01, -6.061917745350e-02, 2.777016064863e-01, -3.581962971027e-01, -1.779651600840e+00],
    "mixed":    [-4.117083535453e+00, -1.744289435038e+00, -8.343452623714e-01, 4.070163490573e+00, 1.349603294952e-01, -3.248938319610e-01, 3.697797732217e-01, 3.037899322158e-01, -1.366742097202e-01, -1.679595301960e+00],
    "density":  [-5.095345177778e+00, -2.071119179924e+00, -2.372794235571e-01, 5.124518164502e+00, 3.894744671716e-02, -4.395745635203e-01, 6.489470953773e-01, 5.182772547490e-01, 2.509904003564e-01, -2.063147445341e+00],
}
COEF_REC = {  # (A0..A5, B0..B5) da recorrência ancorada
    "velocity": ([2.788500117919e-01, -2.966698348018e-01, -1.061539126617e-01, 1.784871200310e-02, 1.060782523752e-01, 3.713049033313e-05],
                 [1.000243598076e+00, 3.616475823146e-03, -3.465008822581e-03, -3.308661337653e-04, -3.357059733527e-03, 2.081862845003e-03]),
    "mixed":    ([1.086007074114e+00, -1.141165207818e+00, -4.412618468854e-01, 5.598732693087e-02, 4.404555832297e-01, 2.590370140402e-04],
                 [9.974431290065e-01, 4.689078443118e-02, -3.752382443014e-02, 1.318779478615e-02, -3.681783800256e-02, 2.182297836168e-02]),
}
F_REGIME = {"velocity": 0.0, "mixed": 0.5, "density": 1.0}
REGIME_SISTEMA = {"Liu et al. 2000": "mixed", "Marston 2025": "density"}


def P_res_regime(alpha, beta, delta, regime):
    c = COEF_G_REGIME[regime]
    la, lb = np.log(alpha), np.log(beta)
    g = (c[0] + c[3]*delta + c[9]*delta**2
         + la*(c[1] + c[6]*delta + c[4]*la + c[5]*lb)
         + lb*(c[2] + c[8]*delta + c[7]*lb))
    return 1.0/(1.0 + np.exp(-g))


def linhas_recorrencia(x1, alpha, eta, delta, regime, x_max):
    """Posições x_M geradas pela recorrência a partir de x1, até x_max."""
    if regime not in COEF_REC:
        return np.array([x1])
    A, B = (np.asarray(v, dtype=float) for v in COEF_REC[regime])
    termos = np.array([1.0, eta, delta, eta**2, eta*delta, delta**2])
    x_ref = np.sqrt(3.0*alpha**(2.0 - F_REGIME[regime]))
    seq = [float(x1)]
    while len(seq) < 1000:
        nxt = x_ref*(A @ termos) + (B @ termos)*seq[-1]
        if not np.isfinite(nxt) or nxt <= seq[-1] or nxt > x_max:
            break
        seq.append(float(nxt))
    return np.array(seq)


for name, r in results.items():
    regime = REGIME_SISTEMA.get(name, "velocity")
    r["regime"] = regime
    r["P_res"] = P_res_regime(r["alpha"], r["beta"], r["delta"], regime)
    picos = r["peaks_yM"] if "peaks_yM" in r else r["peaks_wM"]
    eixo = r["yM_vals"] if "yM_vals" in r else r["wM_vals"]
    escala = 1.0 + r["delta"]          # eixo do gráfico = (1+delta) x_M
    if len(picos) == 0:
        r["linhas_rec"] = np.array([])
    else:
        todas = escala*linhas_recorrencia(picos[0]/escala, r["alpha"], r["eta"], r["delta"],
                                          regime, eixo.max()/escala)
        # mantém só a posição da recorrência mais próxima de cada pico resolvido
        # (as posições sem pico correspondente no espectro não são desenhadas)
        r["linhas_rec_todas"] = todas
        r["linhas_rec"] = np.unique([todas[np.argmin(np.abs(todas - p))] for p in picos])
        desvio = np.array([np.min(np.abs(todas - p)) for p in picos])/escala
        print(f"{name}: {len(todas)} posições da recorrência, {len(r['linhas_rec'])} desenhadas "
              f"({len(picos)} picos); desvio máximo {desvio.max():.4f} em x_M, "
              f"desvio relativo médio {100*np.mean(desvio/(np.asarray(picos)/escala)):.1f}%")


def plot_validation_case(
    name,
    r
):

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )


    # --------------------------------------------------------
    # Posições previstas pela relação de recorrência
    # --------------------------------------------------------
    for yp in r["linhas_rec"]:

        ax.axvline(
            yp,
            color="0.7",
            linestyle=":",
            linewidth=1.2,
            zorder=1,
        )


    # --------------------------------------------------------
    # Energia interna
    # --------------------------------------------------------
    ax.plot(
        r["yM_vals"],
        r["W_total"],
        color=COLORS[0],
        linestyle=LINESTYLES[0],
        linewidth=2.0,
        zorder=2,
    )


    # --------------------------------------------------------
    # Marcadores dos picos
    # --------------------------------------------------------
    if len(
        r["peaks_yM"]
    ) > 0:

        ax.scatter(
            r["peaks_yM"],
            r["peaks_W"],
            s=24,
            color=COLORS[0],
            edgecolors="white",
            linewidths=0.45,
            zorder=5,
        )


    # --------------------------------------------------------
    # Valor da literatura de Marston (2025)
    # --------------------------------------------------------
    if (
        name == "Marston 2025"
        and r["yM_known"] is not None
    ):

        ax.axvline(
            r["yM_known"],
            color=COLORS[1],
            linestyle="--",
            linewidth=2.0,
            ymin=0.0,
            ymax=0.55,
            zorder=4,
        )


    # ========================================================
    # ESCALAS
    # ========================================================
    ax.set_yscale(
        "log"
    )

    ax.set_xlim(
        0.0,
        r["yM_vals"].max(),
    )

    ax.set_ylim(
        *r["ylim"]
    )


    # --------------------------------------------------------
    # Ticks do eixo horizontal
    # --------------------------------------------------------
    ax.xaxis.set_major_locator(
        FixedLocator([
            0.25,
            0.50,
            0.75,
            1.00,
            1.25
        ])
    )

    ax.xaxis.set_major_formatter(
        FuncFormatter(
            format_x_tick
        )
    )


    # --------------------------------------------------------
    # Ticks do eixo vertical
    # --------------------------------------------------------
    ax.yaxis.set_major_locator(
        FixedLocator(
            r["y_ticks"]
        )
    )


    # --------------------------------------------------------
    # Remoção dos ticks secundários
    # --------------------------------------------------------
    ax.minorticks_off()

    ax.yaxis.set_minor_formatter(
        NullFormatter()
    )


    # ========================================================
    # EIXO HORIZONTAL
    # ========================================================
    ax.set_xlabel(
        r"$y_M=(1+\delta)\,x_M$",
        fontsize=LABEL_FONTSIZE,
        labelpad=8,
    )


    # ========================================================
    # TICKS
    # ========================================================
    ax.tick_params(
        direction="in",
        length=5,
        width=1.2,
        labelsize=TICK_FONTSIZE,
        pad=6,
    )


    # ========================================================
    # MOLDURA
    # ========================================================
    for spine in ax.spines.values():

        spine.set_linewidth(
            1.8
        )

        spine.set_edgecolor(
            "black"
        )


    ax.grid(
        False
    )


    # ========================================================
    # LEGENDA
    # ========================================================
    handles = []


    # --------------------------------------------------------
    # Marston (2025)
    # --------------------------------------------------------
    if (
        name == "Marston 2025"
        and r["yM_known"] is not None
    ):

        yM_lit_text = fmt_decimal_math(
            r["yM_known"],
            decimals=4
        )

        h_author = Line2D(
            [0],
            [0],
            color=COLORS[1],
            linestyle="--",
            linewidth=2.0,
            label=(
                author_labels[name]
                + rf": $y_M^{{\mathrm{{lit.}}}}"
                + rf"={yM_lit_text}$"
            ),
        )

        handles.append(
            h_author
        )


    # --------------------------------------------------------
    # Liu et al. (2000)
    # --------------------------------------------------------
    else:

        h_author = Rectangle(
            (0, 0),
            0,
            0,
            fill=False,
            edgecolor="none",
            visible=False,
            label=author_labels[name],
        )

        handles.append(
            h_author
        )


    # --------------------------------------------------------
    # alpha e beta
    # --------------------------------------------------------
    h_text1 = Rectangle(
        (0, 0),
        0,
        0,
        fill=False,
        edgecolor="none",
        visible=False,
        label=(
            r"$\alpha="
            + fmt_latex(
                r["alpha"]
            )
            + r",\ \beta="
            + fmt_latex(
                r["beta"]
            )
            + r"$"
        ),
    )


    # --------------------------------------------------------
    # delta e probabilidade de ressonância
    # --------------------------------------------------------
    delta_text = fmt_decimal_math(
        r["delta"],
        decimals=3
    )

    pres_text = fmt_decimal_math(
        r["P_res"],
        decimals=4
    )

    h_text2 = Rectangle(
        (0, 0),
        0,
        0,
        fill=False,
        edgecolor="none",
        visible=False,
        label=(
            rf"$\delta={delta_text},\ "
            rf"P_{{\mathrm{{res}}}}={pres_text}$"
        ),
    )


    handles.extend([
        h_text1,
        h_text2,
    ])


    # --------------------------------------------------------
    # Box da legenda
    # --------------------------------------------------------
    leg = ax.legend(
        handles=handles,
        loc="upper right",
        fontsize=LEGEND_FONTSIZE,
        frameon=True,
        borderpad=0.5,
        labelspacing=0.35,
        handlelength=(
            2.0
            if name == "Marston 2025"
            else 0
        ),
    )


    leg.get_frame().set(
        edgecolor="black",
        facecolor="white",
        linewidth=2.0,
        alpha=1.0,
    )


    # ========================================================
    # AJUSTES FINAIS
    # ========================================================
    fig.subplots_adjust(
        left=0.20,
        right=0.97,
        bottom=0.18,
        top=0.96,
    )


    # ========================================================
    # SALVAMENTO
    # ========================================================
    fig.savefig(
        f"{r['fname']}.png",
        dpi=600,
        bbox_inches="tight",
        pad_inches=0.05,
    )

    fig.savefig(
        f"{r['fname']}.pdf",
        bbox_inches="tight",
        pad_inches=0.05,
    )


    plt.show()

    plt.close(
        fig
    )


# ============================================================
# GERAR FIGURAS
# ============================================================
for name, r in results.items():

    plot_validation_case(
        name,
        r,
    )


print(
    "Salvo:",
    [
        r["fname"]
        + ".png/.pdf"
        for r in results.values()
    ],
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, NullFormatter
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from scipy.signal import find_peaks

# ============================================================
# Assume já implementados e importados:
#   coeffs, FA, BA, Fjj, Fyy, Fjy, Bjj, Byy, Bjy, get_lmax_core_shell
#   W_core_parts_vs_xM, WT_core_vs_xM, W_shell_parts_vs_xM, WT_shell_vs_xM
# ============================================================

LMAX = 10

# ============================================================
# ESTÉTICA
# ============================================================
COLORS = ["#0f223d", "#810000", "black"]
LINESTYLES = ["-", "--", "-."]
FIGSIZE = (6.5, 5.0)

COEF_G = dict(
    c0=-3.536863223462e+00,
    c1=-1.472216155442e+00,
    c2=-1.163130944691e+00,
    c3=4.246050955653e+00,
    c4=1.694216135501e-01,
    c5=-2.154035917812e-01,
    c6=-6.061917745350e-02,
    c7=2.777016064863e-01,
    c8=-3.581962971027e-01,
    c9=-1.779651600840e+00
)

def g_velocity(alpha, beta, delta):
    la, lb = np.log(alpha), np.log(beta)
    c = COEF_G

    return (
        c["c0"]
        + c["c3"] * delta
        + c["c9"] * delta**2
        + la * (
            c["c1"]
            + c["c6"] * delta
            + c["c4"] * la
            + c["c5"] * lb
        )
        + lb * (
            c["c2"]
            + c["c8"] * delta
            + c["c7"] * lb
        )
    )

def P_res(alpha, beta, delta):
    return 1.0 / (1.0 + np.exp(-g_velocity(alpha, beta, delta)))

def fmt_latex(val):
    """Formata números em notação LaTeX."""
    if val == 0:
        return "0"

    exp = int(np.floor(np.log10(abs(val))))

    if -2 <= exp <= 2:
        return f"{val:.3g}"

    mantissa = val / (10**exp)

    if np.isclose(mantissa, 1.0):
        return f"10^{{{exp}}}"

    if np.isclose(mantissa, -1.0):
        return f"-10^{{{exp}}}"

    return f"{mantissa:.1f} \\times 10^{{{exp}}}"

# ============================================================
# SISTEMAS FÍSICOS
# alpha = Z_A/Z_M
# beta  = Z_B/Z_M
# ============================================================
systems = {
    "Liu et al. 2000": dict(
        rho0=1.18e3,
        c0=2539.52,
        rho1=11.6e3,
        c1=2493.09,
        rho2=1.3e3,
        c2=22.8709,
        delta=0.4200,
        yM_known=None,
        fname="validation_liu2000",
        y_ticks=np.array([1e-7, 1e-3, 1e1, 1e5, 1e9]),
        ylim=(1e-7, 1e10),
    ),

    "Marston 2025": dict(
        rho0=1.22,
        c0=343.0,
        rho1=1.22,
        c1=343.0,
        rho2=841.8 * 1.22,
        c2=4.519 * 343.0,
        delta=(1 - 0.7) / 0.7,
        yM_known=0.1554,
        fname="validation_marston2025",
        y_ticks=np.array([1e-15, 1e-11, 1e-7, 1e-3, 1e1]),
        ylim=(1e-15, 1e2),
    ),
}

author_labels = {
    "Liu et al. 2000": "Liu et al. (2000)",
    "Marston 2025": "Marston (2025)",
}

# ============================================================
# CÁLCULO
# ============================================================
xM_vals = np.linspace(1e-4, 0.75, 1000)
I0_over_A0sq = 1.0

results = {}

for name, p in systems.items():

    rho0 = p["rho0"]
    rho1 = p["rho1"]
    rho2 = p["rho2"]

    c0 = p["c0"]
    c1 = p["c1"]
    c2 = p["c2"]

    delta = p["delta"]

    alpha = (rho1 * c1) / (rho0 * c0)
    beta = (rho2 * c2) / (rho0 * c0)

    eta = (alpha - beta) / (alpha + beta)

    WT_core = WT_core_vs_xM(
        xM_vals,
        delta,
        rho0,
        rho1,
        rho2,
        c0,
        c1,
        c2,
        lmax_core=LMAX,
        lmax_cap=LMAX,
    )

    WT_shell = WT_shell_vs_xM(
        xM_vals,
        delta,
        rho0,
        rho1,
        rho2,
        c0,
        c1,
        c2,
        lmax_shell=LMAX,
        lmax_cap=LMAX,
    )

    Z_A = rho1 * c1
    Z_B = rho2 * c2

    W_total = 2 * I0_over_A0sq * (
        WT_core / Z_A
        + WT_shell / Z_B
    )

    wM_vals = (1 + delta) * xM_vals

    valid = np.isfinite(W_total) & (W_total > 0)

    logW = np.full_like(W_total, -np.inf)
    logW[valid] = np.log10(W_total[valid])

    idx, _ = find_peaks(
        logW,
        prominence=0.75,
    )

    peaks_wM = wM_vals[idx]
    peaks_W = W_total[idx]

    results[name] = dict(
        alpha=alpha,
        beta=beta,
        eta=eta,
        delta=delta,
        wM_vals=wM_vals,
        W_total=W_total,
        peaks_wM=peaks_wM,
        peaks_W=peaks_W,
        P_res=P_res(alpha, beta, delta),
        yM_known=p["yM_known"],
        fname=p["fname"],
        y_ticks=p["y_ticks"],
        ylim=p["ylim"],
    )

# ============================================================
# GRÁFICO
# ============================================================
# ============================================================
# REGIME DE CADA SISTEMA E LINHAS CINZA PELA RELAÇÃO DE RECORRÊNCIA
#   Liu: o contraste do núcleo vem da densidade e o da casca, da velocidade -> regime misto (f = 0,5)
#   Marston: meio e núcleo de ar, contraste dominado pela densidade -> regime de densidade (f = 1)
#   demais sistemas: regime de velocidade (f = 0)
#   P_res usa os coeficientes c0..c9 do regime (dataset coreshell_final/resultados_mix).
#   Linhas cinza: x_(n+1) = x_ref A(eta,delta) + B(eta,delta) x_n, x_ref = sqrt(3 alpha^(2-f)),
#   geradas a partir do primeiro pico resolvido; o regime de densidade não tem recorrência,
#   então nele a única linha é a do primeiro pico.
# ============================================================
COEF_G_REGIME = {
    "velocity": [-3.536863223462e+00, -1.472216155442e+00, -1.163130944691e+00, 4.246050955653e+00, 1.694216135501e-01, -2.154035917812e-01, -6.061917745350e-02, 2.777016064863e-01, -3.581962971027e-01, -1.779651600840e+00],
    "mixed":    [-4.117083535453e+00, -1.744289435038e+00, -8.343452623714e-01, 4.070163490573e+00, 1.349603294952e-01, -3.248938319610e-01, 3.697797732217e-01, 3.037899322158e-01, -1.366742097202e-01, -1.679595301960e+00],
    "density":  [-5.095345177778e+00, -2.071119179924e+00, -2.372794235571e-01, 5.124518164502e+00, 3.894744671716e-02, -4.395745635203e-01, 6.489470953773e-01, 5.182772547490e-01, 2.509904003564e-01, -2.063147445341e+00],
}
COEF_REC = {  # (A0..A5, B0..B5) da recorrência ancorada
    "velocity": ([2.788500117919e-01, -2.966698348018e-01, -1.061539126617e-01, 1.784871200310e-02, 1.060782523752e-01, 3.713049033313e-05],
                 [1.000243598076e+00, 3.616475823146e-03, -3.465008822581e-03, -3.308661337653e-04, -3.357059733527e-03, 2.081862845003e-03]),
    "mixed":    ([1.086007074114e+00, -1.141165207818e+00, -4.412618468854e-01, 5.598732693087e-02, 4.404555832297e-01, 2.590370140402e-04],
                 [9.974431290065e-01, 4.689078443118e-02, -3.752382443014e-02, 1.318779478615e-02, -3.681783800256e-02, 2.182297836168e-02]),
}
F_REGIME = {"velocity": 0.0, "mixed": 0.5, "density": 1.0}
REGIME_SISTEMA = {"Liu et al. 2000": "mixed", "Marston 2025": "density"}


def P_res_regime(alpha, beta, delta, regime):
    c = COEF_G_REGIME[regime]
    la, lb = np.log(alpha), np.log(beta)
    g = (c[0] + c[3]*delta + c[9]*delta**2
         + la*(c[1] + c[6]*delta + c[4]*la + c[5]*lb)
         + lb*(c[2] + c[8]*delta + c[7]*lb))
    return 1.0/(1.0 + np.exp(-g))


def linhas_recorrencia(x1, alpha, eta, delta, regime, x_max):
    """Posições x_M geradas pela recorrência a partir de x1, até x_max."""
    if regime not in COEF_REC:
        return np.array([x1])
    A, B = (np.asarray(v, dtype=float) for v in COEF_REC[regime])
    termos = np.array([1.0, eta, delta, eta**2, eta*delta, delta**2])
    x_ref = np.sqrt(3.0*alpha**(2.0 - F_REGIME[regime]))
    seq = [float(x1)]
    while len(seq) < 1000:
        nxt = x_ref*(A @ termos) + (B @ termos)*seq[-1]
        if not np.isfinite(nxt) or nxt <= seq[-1] or nxt > x_max:
            break
        seq.append(float(nxt))
    return np.array(seq)


for name, r in results.items():
    regime = REGIME_SISTEMA.get(name, "velocity")
    r["regime"] = regime
    r["P_res"] = P_res_regime(r["alpha"], r["beta"], r["delta"], regime)
    picos = r["peaks_yM"] if "peaks_yM" in r else r["peaks_wM"]
    eixo = r["yM_vals"] if "yM_vals" in r else r["wM_vals"]
    escala = 1.0 + r["delta"]          # eixo do gráfico = (1+delta) x_M
    if len(picos) == 0:
        r["linhas_rec"] = np.array([])
    else:
        todas = escala*linhas_recorrencia(picos[0]/escala, r["alpha"], r["eta"], r["delta"],
                                          regime, eixo.max()/escala)
        # mantém só a posição da recorrência mais próxima de cada pico resolvido
        # (as posições sem pico correspondente no espectro não são desenhadas)
        r["linhas_rec_todas"] = todas
        r["linhas_rec"] = np.unique([todas[np.argmin(np.abs(todas - p))] for p in picos])
        desvio = np.array([np.min(np.abs(todas - p)) for p in picos])/escala
        print(f"{name}: {len(todas)} posições da recorrência, {len(r['linhas_rec'])} desenhadas "
              f"({len(picos)} picos); desvio máximo {desvio.max():.4f} em x_M, "
              f"desvio relativo médio {100*np.mean(desvio/(np.asarray(picos)/escala)):.1f}%")


def plot_validation_case(name, r):

    fig, ax = plt.subplots(figsize=FIGSIZE)

    # --------------------------------------------------------
    # Posições previstas pela relação de recorrência
    # --------------------------------------------------------
    for yp in r["linhas_rec"]:
        ax.axvline(
            yp,
            color="0.7",
            linestyle=":",
            linewidth=1.2,
            zorder=1,
        )

    # --------------------------------------------------------
    # Energia interna
    # --------------------------------------------------------
    ax.plot(
        r["wM_vals"],
        r["W_total"],
        color=COLORS[0],
        linestyle=LINESTYLES[0],
        linewidth=2.0,
        zorder=2,
    )

    # --------------------------------------------------------
    # Marcadores dos picos
    # --------------------------------------------------------
    if len(r["peaks_wM"]) > 0:
        ax.scatter(
            r["peaks_wM"],
            r["peaks_W"],
            s=24,
            color=COLORS[0],
            edgecolors="white",
            linewidths=0.45,
            zorder=5,
        )

    # --------------------------------------------------------
    # Valor da literatura de Marston (2025)
    # --------------------------------------------------------
    if name == "Marston 2025" and r["yM_known"] is not None:
        ax.axvline(
            r["yM_known"],
            color=COLORS[1],
            linestyle="--",
            linewidth=2.0,
            ymin=0.0,
            ymax=0.55,
            zorder=4,
        )

    # --------------------------------------------------------
    # Escalas
    # --------------------------------------------------------
    ax.set_yscale("log")

    ax.set_xlim(
        0.0,
        r["wM_vals"].max(),
    )

    ax.set_ylim(
        *r["ylim"]
    )

    ax.xaxis.set_major_locator(
        FixedLocator(
            [0.25, 0.50, 0.75, 1.00, 1.25]
        )
    )

    ax.yaxis.set_major_locator(
        FixedLocator(
            r["y_ticks"]
        )
    )

    ax.minorticks_off()
    ax.yaxis.set_minor_formatter(
        NullFormatter()
    )

    ax.set_xlabel(
        r"$w_M=(1+\delta)\,x_M$",
        fontsize=20,
        labelpad=8,
    )

    ax.tick_params(
        direction="in",
        length=5,
        width=1.2,
        labelsize=16,
        pad=6,
    )

    for spine in ax.spines.values():
        spine.set_linewidth(1.8)
        spine.set_edgecolor("black")

    ax.grid(False)

    # ========================================================
    # LEGENDA
    # ========================================================
    handles = []

    # --------------------------------------------------------
    # MARSTON:
    # a própria linha vermelha tracejada aparece ao lado
    # de "Marston (2025)"
    # --------------------------------------------------------
    if name == "Marston 2025" and r["yM_known"] is not None:

        h_author = Line2D(
            [0],
            [0],
            color=COLORS[1],
            linestyle="--",
            linewidth=2.0,
            label=(
                author_labels[name]
                + r": $w_M^{\mathrm{lit.}}=%.4f$"
                % r["yM_known"]
            ),
        )

        handles.append(
            h_author
        )

    # --------------------------------------------------------
    # LIU:
    # somente o nome, pois não existe linha vermelha de
    # referência neste caso
    # --------------------------------------------------------
    else:

        h_author = Rectangle(
            (0, 0),
            0,
            0,
            fill=False,
            edgecolor="none",
            visible=False,
            label=author_labels[name],
        )

        handles.append(
            h_author
        )

    # --------------------------------------------------------
    # alpha e beta
    # --------------------------------------------------------
    h_text1 = Rectangle(
        (0, 0),
        0,
        0,
        fill=False,
        edgecolor="none",
        visible=False,
        label=(
            r"$\alpha=%s,\ \beta=%s$"
            % (
                fmt_latex(r["alpha"]),
                fmt_latex(r["beta"]),
            )
        ),
    )

    # --------------------------------------------------------
    # delta e probabilidade
    # --------------------------------------------------------
    h_text2 = Rectangle(
        (0, 0),
        0,
        0,
        fill=False,
        edgecolor="none",
        visible=False,
        label=(
            r"$\delta=%.3f,\ P_{\mathrm{res}}=%.4f$"
            % (
                r["delta"],
                r["P_res"],
            )
        ),
    )

    handles.extend(
        [
            h_text1,
            h_text2,
        ]
    )

    # --------------------------------------------------------
    # Box
    # --------------------------------------------------------
    leg = ax.legend(
        handles=handles,
        loc="upper right",
        fontsize=15,
        frameon=True,
        borderpad=0.5,
        labelspacing=0.35,
        handlelength=2.0 if name == "Marston 2025" else 0,
    )

    leg.get_frame().set(
        edgecolor="black",
        facecolor="white",
        linewidth=2.0,
        alpha=1.0,
    )

    # ========================================================
    # AJUSTES FINAIS
    # ========================================================
    fig.subplots_adjust(
        left=0.20,
        right=0.97,
        bottom=0.18,
        top=0.96,
    )

    fig.savefig(
        f"{r['fname']}.png",
        dpi=600,
        bbox_inches="tight",
        pad_inches=0.05,
    )

    fig.savefig(
        f"{r['fname']}.pdf",
        bbox_inches="tight",
        pad_inches=0.05,
    )

    plt.show()
    plt.close(fig)

# ============================================================
# GERAR FIGURAS
# ============================================================
for name, r in results.items():
    plot_validation_case(
        name,
        r,
    )

print(
    "Saved:",
    [
        r["fname"] + ".png/.pdf"
        for r in results.values()
    ],
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

# ============================================================
# Assume already implemented and imported:
#   coeffs, FA, BA, Fjj, Fyy, Fjy, Bjj, Byy, Bjy, get_lmax_core_shell
#   W_core_parts_vs_xM, WT_core_vs_xM, W_shell_parts_vs_xM, WT_shell_vs_xM
# ============================================================

plt.rcParams.update({
    "font.family": "serif",
    "mathtext.fontset": "cm",
    "font.size": 11,
    "axes.linewidth": 1.0,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "legend.frameon": True,
    "legend.fontsize": 8,
})

LMAX = 10

COEF_G = dict(c0=-3.536863223462e+00, c1=-1.472216155442e+00, c2=-1.163130944691e+00,
              c3=4.246050955653e+00, c4=1.694216135501e-01, c5=-2.154035917812e-01,
              c6=-6.061917745350e-02, c7=2.777016064863e-01, c8=-3.581962971027e-01,
              c9=-1.779651600840e+00)

def g_velocity(alpha, beta, delta):
    la, lb = np.log(alpha), np.log(beta)
    c = COEF_G
    return (c['c0'] + c['c3']*delta + c['c9']*delta**2
            + la*(c['c1'] + c['c6']*delta + c['c4']*la + c['c5']*lb)
            + lb*(c['c2'] + c['c8']*delta + c['c7']*lb))

def P_res(alpha, beta, delta):
    return 1.0/(1.0 + np.exp(-g_velocity(alpha, beta, delta)))


# ============================================================
# Physical systems (alpha = Z_A/Z_M, beta = Z_B/Z_M)
# ============================================================
systems = {
    "Liu et al. 2000": dict(
        rho0=1.18e3,  c0=2539.52,
        rho1=11.6e3,  c1=2493.09,
        rho2=1.3e3,   c2=22.8709,
        delta=0.4200,
        yM_known=None,
        fname="validation_liu2000.png",
    ),
    "Marston 2025": dict(
        rho0=1.22,    c0=343.0,
        rho1=1.22,    c1=343.0,
        rho2=841.8*1.22, c2=4.519*343.0,
        delta=(1-0.7)/0.7,
        yM_known=0.1554,
        fname="validation_marston2025.png",
    ),
}

xM_vals = np.linspace(1e-4, 1.0, 4000)
I0_over_A0sq = 1.0

results = {}
for name, p in systems.items():
    rho0, rho1, rho2 = p["rho0"], p["rho1"], p["rho2"]
    c0, c1, c2 = p["c0"], p["c1"], p["c2"]
    delta = p["delta"]

    alpha = (rho1*c1)/(rho0*c0)
    beta  = (rho2*c2)/(rho0*c0)
    eta   = (alpha - beta)/(alpha + beta)

    WT_core  = WT_core_vs_xM(xM_vals, delta, rho0, rho1, rho2, c0, c1, c2,
                              lmax_core=LMAX, lmax_cap=LMAX)
    WT_shell = WT_shell_vs_xM(xM_vals, delta, rho0, rho1, rho2, c0, c1, c2,
                               lmax_shell=LMAX, lmax_cap=LMAX)

    Z_A, Z_B = rho1*c1, rho2*c2
    W_total = 2*I0_over_A0sq*(WT_core/Z_A + WT_shell/Z_B)
    yM_vals = (1 + delta) * xM_vals

    valid = np.isfinite(W_total) & (W_total > 0)
    logW = np.full_like(W_total, -np.inf)
    logW[valid] = np.log10(W_total[valid])
    idx, _ = find_peaks(logW, prominence=0.75)
    peaks_yM = yM_vals[idx]

    results[name] = dict(alpha=alpha, beta=beta, eta=eta, delta=delta,
                          yM_vals=yM_vals, W_total=W_total, peaks_yM=peaks_yM,
                          P_res=P_res(alpha, beta, delta), yM_known=p["yM_known"],
                          fname=p["fname"])

# ============================================================
# Plot -- one separate, self-contained figure per system
# each with its OWN y-scale (they differ by many orders of magnitude)
# ============================================================
author_labels = {
    "Liu et al. 2000": "Liu et al. (2000)",
    "Marston 2025": "Marston (2025)",
}

# ============================================================
# REGIME DE CADA SISTEMA E LINHAS CINZA PELA RELAÇÃO DE RECORRÊNCIA
#   Liu: o contraste do núcleo vem da densidade e o da casca, da velocidade -> regime misto (f = 0,5)
#   Marston: meio e núcleo de ar, contraste dominado pela densidade -> regime de densidade (f = 1)
#   demais sistemas: regime de velocidade (f = 0)
#   P_res usa os coeficientes c0..c9 do regime (dataset coreshell_final/resultados_mix).
#   Linhas cinza: x_(n+1) = x_ref A(eta,delta) + B(eta,delta) x_n, x_ref = sqrt(3 alpha^(2-f)),
#   geradas a partir do primeiro pico resolvido; o regime de densidade não tem recorrência,
#   então nele a única linha é a do primeiro pico.
# ============================================================
COEF_G_REGIME = {
    "velocity": [-3.536863223462e+00, -1.472216155442e+00, -1.163130944691e+00, 4.246050955653e+00, 1.694216135501e-01, -2.154035917812e-01, -6.061917745350e-02, 2.777016064863e-01, -3.581962971027e-01, -1.779651600840e+00],
    "mixed":    [-4.117083535453e+00, -1.744289435038e+00, -8.343452623714e-01, 4.070163490573e+00, 1.349603294952e-01, -3.248938319610e-01, 3.697797732217e-01, 3.037899322158e-01, -1.366742097202e-01, -1.679595301960e+00],
    "density":  [-5.095345177778e+00, -2.071119179924e+00, -2.372794235571e-01, 5.124518164502e+00, 3.894744671716e-02, -4.395745635203e-01, 6.489470953773e-01, 5.182772547490e-01, 2.509904003564e-01, -2.063147445341e+00],
}
COEF_REC = {  # (A0..A5, B0..B5) da recorrência ancorada
    "velocity": ([2.788500117919e-01, -2.966698348018e-01, -1.061539126617e-01, 1.784871200310e-02, 1.060782523752e-01, 3.713049033313e-05],
                 [1.000243598076e+00, 3.616475823146e-03, -3.465008822581e-03, -3.308661337653e-04, -3.357059733527e-03, 2.081862845003e-03]),
    "mixed":    ([1.086007074114e+00, -1.141165207818e+00, -4.412618468854e-01, 5.598732693087e-02, 4.404555832297e-01, 2.590370140402e-04],
                 [9.974431290065e-01, 4.689078443118e-02, -3.752382443014e-02, 1.318779478615e-02, -3.681783800256e-02, 2.182297836168e-02]),
}
F_REGIME = {"velocity": 0.0, "mixed": 0.5, "density": 1.0}
REGIME_SISTEMA = {"Liu et al. 2000": "mixed", "Marston 2025": "density"}


def P_res_regime(alpha, beta, delta, regime):
    c = COEF_G_REGIME[regime]
    la, lb = np.log(alpha), np.log(beta)
    g = (c[0] + c[3]*delta + c[9]*delta**2
         + la*(c[1] + c[6]*delta + c[4]*la + c[5]*lb)
         + lb*(c[2] + c[8]*delta + c[7]*lb))
    return 1.0/(1.0 + np.exp(-g))


def linhas_recorrencia(x1, alpha, eta, delta, regime, x_max):
    """Posições x_M geradas pela recorrência a partir de x1, até x_max."""
    if regime not in COEF_REC:
        return np.array([x1])
    A, B = (np.asarray(v, dtype=float) for v in COEF_REC[regime])
    termos = np.array([1.0, eta, delta, eta**2, eta*delta, delta**2])
    x_ref = np.sqrt(3.0*alpha**(2.0 - F_REGIME[regime]))
    seq = [float(x1)]
    while len(seq) < 1000:
        nxt = x_ref*(A @ termos) + (B @ termos)*seq[-1]
        if not np.isfinite(nxt) or nxt <= seq[-1] or nxt > x_max:
            break
        seq.append(float(nxt))
    return np.array(seq)


for name, r in results.items():
    regime = REGIME_SISTEMA.get(name, "velocity")
    r["regime"] = regime
    r["P_res"] = P_res_regime(r["alpha"], r["beta"], r["delta"], regime)
    picos = r["peaks_yM"] if "peaks_yM" in r else r["peaks_wM"]
    eixo = r["yM_vals"] if "yM_vals" in r else r["wM_vals"]
    escala = 1.0 + r["delta"]          # eixo do gráfico = (1+delta) x_M
    if len(picos) == 0:
        r["linhas_rec"] = np.array([])
    else:
        todas = escala*linhas_recorrencia(picos[0]/escala, r["alpha"], r["eta"], r["delta"],
                                          regime, eixo.max()/escala)
        # mantém só a posição da recorrência mais próxima de cada pico resolvido
        # (as posições sem pico correspondente no espectro não são desenhadas)
        r["linhas_rec_todas"] = todas
        r["linhas_rec"] = np.unique([todas[np.argmin(np.abs(todas - p))] for p in picos])
        desvio = np.array([np.min(np.abs(todas - p)) for p in picos])/escala
        print(f"{name}: {len(todas)} posições da recorrência, {len(r['linhas_rec'])} desenhadas "
              f"({len(picos)} picos); desvio máximo {desvio.max():.4f} em x_M, "
              f"desvio relativo médio {100*np.mean(desvio/(np.asarray(picos)/escala)):.1f}%")


for name, r in results.items():
    fig, ax = plt.subplots(figsize=(3.4, 3.0))

    ax.semilogy(r["yM_vals"], r["W_total"], color="k", lw=1.1, zorder=2)

    # gray dotted lines: positions predicted by the recurrence relation
    for yp in r["linhas_rec"]:
        ax.axvline(yp, color="0.7", ls=":", lw=0.8, zorder=1)

    # reference value from the literature: shorter than full height so the
    # gray peak line underneath remains visible
    if r["yM_known"] is not None:
        ax.axvline(r["yM_known"], color="firebrick", ls="--", lw=1.3,
                    ymin=0.0, ymax=0.55, zorder=3,
                    label=r"$y_M^{\mathrm{lit.}}=%.4f$" % r["yM_known"])
        ax.legend(loc="lower right", fontsize=7, framealpha=0.9)

    ax.set_xlabel(r"$y_M = (1+\delta)\,x_M$")
    ax.set_xlim(0, r["yM_vals"].max())

    # each panel keeps its own y-range, scaled to its own data
    finite = r["W_total"][np.isfinite(r["W_total"]) & (r["W_total"] > 0)]
    ymin = 10**(np.floor(np.log10(finite.min())))
    ymax = 10**(np.ceil(np.log10(finite.max())) + 0.5)
    ax.set_ylim(ymin, ymax)

    ax.set_ylabel(r"$\frac{2W_T}{|A|^2}\left(\frac{W_A^{core}}{Z_A}+\frac{W_B^{shell}}{Z_B}\right)$")

    # box with author + fit parameters together
    txt = (author_labels[name] + "\n" +
           r"$\eta \approx %.2f$" % r["eta"] + "\n" +
           r"$\alpha=%.3g,\ \beta=%.3g$" % (r["alpha"], r["beta"]) + "\n" +
           r"$\delta=%.3f,\ P_{\mathrm{res}}=%.4f$" % (r["delta"], r["P_res"]))
    ax.text(0.97, 0.97, txt, transform=ax.transAxes, ha="right", va="top",
             fontsize=7.5, linespacing=1.4,
             bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="k", lw=0.8))

    plt.tight_layout()
    plt.savefig(r["fname"], dpi=600, bbox_inches="tight")
    plt.close(fig)

print("Saved:", [r["fname"] for r in results.values()])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, NullFormatter
from scipy.signal import find_peaks

# ============================================================
# Assume already implemented and imported:
#   coeffs, FA, BA, Fjj, Fyy, Fjy, Bjj, Byy, Bjy, get_lmax_core_shell
#   W_core_parts_vs_xM, WT_core_vs_xM, W_shell_parts_vs_xM, WT_shell_vs_xM
# ============================================================

LMAX = 10

# ---------------- "estética antiga" -- cores, linhas, caixas, ticks ----------------
COLORS = ["#0f223d", "#810000", "black"]
LINESTYLES = ["-", "--", "-."]
FIGSIZE = (6.5, 5.0)

COEF_G = dict(c0=-3.536863223462e+00, c1=-1.472216155442e+00, c2=-1.163130944691e+00,
              c3=4.246050955653e+00, c4=1.694216135501e-01, c5=-2.154035917812e-01,
              c6=-6.061917745350e-02, c7=2.777016064863e-01, c8=-3.581962971027e-01,
              c9=-1.779651600840e+00)

def g_velocity(alpha, beta, delta):
    la, lb = np.log(alpha), np.log(beta)
    c = COEF_G
    return (c['c0'] + c['c3']*delta + c['c9']*delta**2
            + la*(c['c1'] + c['c6']*delta + c['c4']*la + c['c5']*lb)
            + lb*(c['c2'] + c['c8']*delta + c['c7']*lb))

def P_res(alpha, beta, delta):
    return 1.0/(1.0 + np.exp(-g_velocity(alpha, beta, delta)))


# ============================================================
# Physical systems (alpha = Z_A/Z_M, beta = Z_B/Z_M)
# ============================================================
systems = {
    "Liu et al. 2000": dict(
        rho0=1.18e3,  c0=2539.52,
        rho1=11.6e3,  c1=2493.09,
        rho2=1.3e3,   c2=22.8709,
        delta=0.4200,
        yM_known=None,
        fname="validation_liu2000",
        y_ticks=np.array([1e-7, 1e-3, 1e1, 1e5, 1e9]),
        ylim=(1e-7, 1e10),
    ),
    "Marston 2025": dict(
        rho0=1.22,    c0=343.0,
        rho1=1.22,    c1=343.0,
        rho2=841.8*1.22, c2=4.519*343.0,
        delta=(1-0.7)/0.7,
        yM_known=0.1554,
        fname="validation_marston2025",
        y_ticks=np.array([1e-15, 1e-11, 1e-7, 1e-3, 1e1]),
        ylim=(1e-15, 1e2),
    ),
}

author_labels = {
    "Liu et al. 2000": "Liu et al. (2000)",
    "Marston 2025": "Marston (2025)",
}

xM_vals = np.linspace(1e-4, 0.75, 1000)
I0_over_A0sq = 1.0

results = {}
for name, p in systems.items():
    rho0, rho1, rho2 = p["rho0"], p["rho1"], p["rho2"]
    c0, c1, c2 = p["c0"], p["c1"], p["c2"]
    delta = p["delta"]

    alpha = (rho1*c1)/(rho0*c0)
    beta  = (rho2*c2)/(rho0*c0)
    eta   = (alpha - beta)/(alpha + beta)

    WT_core  = WT_core_vs_xM(xM_vals, delta, rho0, rho1, rho2, c0, c1, c2,
                              lmax_core=LMAX, lmax_cap=LMAX)
    WT_shell = WT_shell_vs_xM(xM_vals, delta, rho0, rho1, rho2, c0, c1, c2,
                               lmax_shell=LMAX, lmax_cap=LMAX)

    Z_A, Z_B = rho1*c1, rho2*c2
    W_total = 2*I0_over_A0sq*(WT_core/Z_A + WT_shell/Z_B)
    yM_vals = (1 + delta) * xM_vals

    valid = np.isfinite(W_total) & (W_total > 0)
    logW = np.full_like(W_total, -np.inf)
    logW[valid] = np.log10(W_total[valid])
    idx, _ = find_peaks(logW, prominence=0.75)
    peaks_yM = yM_vals[idx]
    peaks_W  = W_total[idx]

    results[name] = dict(alpha=alpha, beta=beta, eta=eta, delta=delta,
                          yM_vals=yM_vals, W_total=W_total,
                          peaks_yM=peaks_yM, peaks_W=peaks_W,
                          P_res=P_res(alpha, beta, delta),
                          yM_known=p["yM_known"], fname=p["fname"],
                          y_ticks=p["y_ticks"], ylim=p["ylim"])


# ============================================================
# Plot -- estética antiga + linhas de referência (cinza=picos, vermelho=literatura)
# ============================================================
# ============================================================
# REGIME DE CADA SISTEMA E LINHAS CINZA PELA RELAÇÃO DE RECORRÊNCIA
#   Liu: o contraste do núcleo vem da densidade e o da casca, da velocidade -> regime misto (f = 0,5)
#   Marston: meio e núcleo de ar, contraste dominado pela densidade -> regime de densidade (f = 1)
#   demais sistemas: regime de velocidade (f = 0)
#   P_res usa os coeficientes c0..c9 do regime (dataset coreshell_final/resultados_mix).
#   Linhas cinza: x_(n+1) = x_ref A(eta,delta) + B(eta,delta) x_n, x_ref = sqrt(3 alpha^(2-f)),
#   geradas a partir do primeiro pico resolvido; o regime de densidade não tem recorrência,
#   então nele a única linha é a do primeiro pico.
# ============================================================
COEF_G_REGIME = {
    "velocity": [-3.536863223462e+00, -1.472216155442e+00, -1.163130944691e+00, 4.246050955653e+00, 1.694216135501e-01, -2.154035917812e-01, -6.061917745350e-02, 2.777016064863e-01, -3.581962971027e-01, -1.779651600840e+00],
    "mixed":    [-4.117083535453e+00, -1.744289435038e+00, -8.343452623714e-01, 4.070163490573e+00, 1.349603294952e-01, -3.248938319610e-01, 3.697797732217e-01, 3.037899322158e-01, -1.366742097202e-01, -1.679595301960e+00],
    "density":  [-5.095345177778e+00, -2.071119179924e+00, -2.372794235571e-01, 5.124518164502e+00, 3.894744671716e-02, -4.395745635203e-01, 6.489470953773e-01, 5.182772547490e-01, 2.509904003564e-01, -2.063147445341e+00],
}
COEF_REC = {  # (A0..A5, B0..B5) da recorrência ancorada
    "velocity": ([2.788500117919e-01, -2.966698348018e-01, -1.061539126617e-01, 1.784871200310e-02, 1.060782523752e-01, 3.713049033313e-05],
                 [1.000243598076e+00, 3.616475823146e-03, -3.465008822581e-03, -3.308661337653e-04, -3.357059733527e-03, 2.081862845003e-03]),
    "mixed":    ([1.086007074114e+00, -1.141165207818e+00, -4.412618468854e-01, 5.598732693087e-02, 4.404555832297e-01, 2.590370140402e-04],
                 [9.974431290065e-01, 4.689078443118e-02, -3.752382443014e-02, 1.318779478615e-02, -3.681783800256e-02, 2.182297836168e-02]),
}
F_REGIME = {"velocity": 0.0, "mixed": 0.5, "density": 1.0}
REGIME_SISTEMA = {"Liu et al. 2000": "mixed", "Marston 2025": "density"}


def P_res_regime(alpha, beta, delta, regime):
    c = COEF_G_REGIME[regime]
    la, lb = np.log(alpha), np.log(beta)
    g = (c[0] + c[3]*delta + c[9]*delta**2
         + la*(c[1] + c[6]*delta + c[4]*la + c[5]*lb)
         + lb*(c[2] + c[8]*delta + c[7]*lb))
    return 1.0/(1.0 + np.exp(-g))


def linhas_recorrencia(x1, alpha, eta, delta, regime, x_max):
    """Posições x_M geradas pela recorrência a partir de x1, até x_max."""
    if regime not in COEF_REC:
        return np.array([x1])
    A, B = (np.asarray(v, dtype=float) for v in COEF_REC[regime])
    termos = np.array([1.0, eta, delta, eta**2, eta*delta, delta**2])
    x_ref = np.sqrt(3.0*alpha**(2.0 - F_REGIME[regime]))
    seq = [float(x1)]
    while len(seq) < 1000:
        nxt = x_ref*(A @ termos) + (B @ termos)*seq[-1]
        if not np.isfinite(nxt) or nxt <= seq[-1] or nxt > x_max:
            break
        seq.append(float(nxt))
    return np.array(seq)


for name, r in results.items():
    regime = REGIME_SISTEMA.get(name, "velocity")
    r["regime"] = regime
    r["P_res"] = P_res_regime(r["alpha"], r["beta"], r["delta"], regime)
    picos = r["peaks_yM"] if "peaks_yM" in r else r["peaks_wM"]
    eixo = r["yM_vals"] if "yM_vals" in r else r["wM_vals"]
    escala = 1.0 + r["delta"]          # eixo do gráfico = (1+delta) x_M
    if len(picos) == 0:
        r["linhas_rec"] = np.array([])
    else:
        todas = escala*linhas_recorrencia(picos[0]/escala, r["alpha"], r["eta"], r["delta"],
                                          regime, eixo.max()/escala)
        # mantém só a posição da recorrência mais próxima de cada pico resolvido
        # (as posições sem pico correspondente no espectro não são desenhadas)
        r["linhas_rec_todas"] = todas
        r["linhas_rec"] = np.unique([todas[np.argmin(np.abs(todas - p))] for p in picos])
        desvio = np.array([np.min(np.abs(todas - p)) for p in picos])/escala
        print(f"{name}: {len(todas)} posições da recorrência, {len(r['linhas_rec'])} desenhadas "
              f"({len(picos)} picos); desvio máximo {desvio.max():.4f} em x_M, "
              f"desvio relativo médio {100*np.mean(desvio/(np.asarray(picos)/escala)):.1f}%")


def plot_validation_case(name, r):
    fig, ax = plt.subplots(figsize=FIGSIZE)

    # linhas cinza pontilhadas: posições previstas pela relação de recorrência
    for yp in r["linhas_rec"]:
        ax.axvline(yp, color="0.7", linestyle=":", linewidth=1.2, zorder=1)

    # curva de energia
    ax.plot(r["yM_vals"], r["W_total"], color=COLORS[0], linestyle=LINESTYLES[0],
            linewidth=2.0, zorder=2)

    # bolinhas nos picos reais (mesma estética do checkpoint real)
    if len(r["peaks_yM"]) > 0:
        ax.scatter(r["peaks_yM"], r["peaks_W"], s=24, color=COLORS[0],
                    edgecolors="white", linewidths=0.45, zorder=5)

    # linha vermelha tracejada -- valor de referência da literatura (mais curta)
    if r["yM_known"] is not None:
        ax.axvline(r["yM_known"], color=COLORS[1], linestyle="--", linewidth=2.0,
                    ymin=0.0, ymax=0.55, zorder=4,
                    label=r"$y_M^{\mathrm{lit.}}=%.4f$" % r["yM_known"])
        legend = ax.legend(loc="lower right", fontsize=20, frameon=True,
                            handlelength=2.5, borderpad=0.50,
                            labelspacing=0.40, handletextpad=0.65)
        legend.get_frame().set(edgecolor="black", facecolor="white",
                                linewidth=1.8, alpha=1.0)

    ax.set_yscale("log")
    ax.set_xlim(0.0, r["yM_vals"].max())
    ax.set_ylim(*r["ylim"])

    ax.xaxis.set_major_locator(FixedLocator([0.25, 0.50, 0.75, 1.00, 1.25]))
    ax.yaxis.set_major_locator(FixedLocator(r["y_ticks"]))
    ax.minorticks_off()
    ax.yaxis.set_minor_formatter(NullFormatter())

    ax.set_xlabel(r"$w_M=(1+\delta)\,x_M$", fontsize=20, labelpad=8)
    ax.tick_params(direction="in", length=5, width=1.2, labelsize=16, pad=6)

    for spine in ax.spines.values():
        spine.set_linewidth(1.8)
        spine.set_edgecolor("black")

    ax.grid(False)

    # caixa com autor + parametros (sem eta), estetica antiga
    box_txt = (author_labels[name] + "\n" +
               r"$\alpha=%.3g,\ \beta=%.3g$" % (r["alpha"], r["beta"]) + "\n" +
               r"$\delta=%.3f,\ P_{\mathrm{res}}=%.4f$" % (r["delta"], r["P_res"]))
    ax.text(0.965, 0.94, box_txt, transform=ax.transAxes, ha="right", va="top",
            fontsize=16, color="black", linespacing=1.4,
            bbox={"boxstyle": "square,pad=0.4", "facecolor": "white",
                  "edgecolor": "black", "linewidth": 2.0, "alpha": 1.0})

    fig.subplots_adjust(left=0.20, right=0.97, bottom=0.18, top=0.96)

    fig.savefig(f"{r['fname']}.png", dpi=600, bbox_inches="tight", pad_inches=0.05)
    fig.savefig(f"{r['fname']}.pdf", bbox_inches="tight", pad_inches=0.05)
    plt.show()
    plt.close(fig)


for name, r in results.items():
    plot_validation_case(name, r)

print("Saved:", [r["fname"] + ".png/.pdf" for r in results.values()])

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as colors
from matplotlib.colors import LinearSegmentedColormap
from scipy.special import eval_legendre
from scipy.signal import find_peaks

# ============================================================
# Assume already implemented/imported:
#   coeffs, J, Y                                  (boundary-matching + Bessel functions)
#   WT_core_vs_xM, WT_shell_vs_xM                  (for auto peak detection)
# ============================================================

OUT_DIR = "internal_fields_validation_liu_marston"
os.makedirs(OUT_DIR, exist_ok=True)

LMAX_ENERGY = 10   # for peak detection only (energy spectrum)

# ---------------- aesthetic helpers ----------------
def make_zero_white_cmap():
    cmap = LinearSegmentedColormap.from_list(
        'white_blue_red_custom', [(0.0, 'white'), (0.5, '#355F7C'), (1.0, '#990000')], N=512)
    cmap.set_bad('white')
    return cmap

def eta_metric(alpha, beta):
    return (alpha - beta) / (alpha + beta)

def number_tag(value):
    return f'{float(value):g}'.replace('-', 'm').replace('.', 'p')

def scaled_plot_sizes(figsize):
    size = min(figsize)
    return {'title': max(8.0, 2.55*size),
            'tick_length': max(2.4, 0.75*size), 'tick_width': max(0.7, 0.18*size),
            'spine_width': max(0.8, 0.2*size), 'circle_a_width': max(0.9, 0.23*size),
            'circle_b_width': max(1.1, 0.28*size)}

# de volta ao tamanho original 3x3, colorbar original -- só o titulo yM cresce
IMAGE_FIGSIZE = (3.5, 3.5)
CBAR_FIGSIZE = (0.45, 2.75)
FIELD_RESOLUTION = 1000
TITLE_FONTSIZE_YM = 15   # tamanho fixo, maior, só pro texto de yM

# ============================================================
# Physical systems (real materials, alpha = Z_A/Z_M, beta = Z_B/Z_M)
# ============================================================
systems = {
    "liu2000": dict(
        label="Liu et al. (2000)",
        rho0=1.18e3,  c0=2539.52,     # host: epoxy
        rho1=11.6e3,  c1=2493.09,     # core: lead
        rho2=1.3e3,   c2=22.8709,     # shell: silicone rubber (effective fluid)
        delta=0.4200,
    ),
    "marston2025": dict(
        label="Marston (2025)",
        rho0=1.22,    c0=343.0,       # host: air
        rho1=1.22,    c1=343.0,       # core: air
        rho2=841.8*1.22, c2=4.519*343.0,  # shell: glycerol solution
        delta=(1-0.7)/0.7,
    ),
}

# ============================================================
# 1) find the first resonance in yM (outer-radius convention) for each system
# ============================================================
xM_scan = np.linspace(1e-4, 1.0, 1000)

for tag, p in systems.items():
    rho0, rho1, rho2 = p["rho0"], p["rho1"], p["rho2"]
    c0, c1, c2 = p["c0"], p["c1"], p["c2"]
    delta = p["delta"]

    alpha = (rho1*c1)/(rho0*c0)
    beta  = (rho2*c2)/(rho0*c0)
    eta   = eta_metric(alpha, beta)

    WT_core  = WT_core_vs_xM(xM_scan, delta, rho0, rho1, rho2, c0, c1, c2,
                              lmax_core=LMAX_ENERGY, lmax_cap=LMAX_ENERGY)
    WT_shell = WT_shell_vs_xM(xM_scan, delta, rho0, rho1, rho2, c0, c1, c2,
                               lmax_shell=LMAX_ENERGY, lmax_cap=LMAX_ENERGY)
    Z_A, Z_B = rho1*c1, rho2*c2
    W_total = 2.0*(WT_core/Z_A + WT_shell/Z_B)
    yM_scan = (1+delta)*xM_scan

    valid = np.isfinite(W_total) & (W_total > 0)
    logW = np.full_like(W_total, -np.inf)
    logW[valid] = np.log10(W_total[valid])
    idx, _ = find_peaks(logW, prominence=0.75)

    yM_first_peak = float(yM_scan[idx[0]]) if len(idx) > 0 else float(yM_scan[np.nanargmax(W_total)])

    p["alpha"], p["beta"], p["eta"] = alpha, beta, eta
    p["yM_first_peak"] = yM_first_peak
    p["xM_first_peak"] = yM_first_peak/(1+delta)   # convert to core-radius (xM) convention
    p["xM_at_yM1"]     = 1.0/(1+delta)              # yM = 1  ->  xM = 1/(1+delta)

    print(f"{tag}: alpha={alpha:.4g} beta={beta:.4g} delta={delta:.4f} eta={eta:+.4f} "
          f"| first resonance yM={yM_first_peak:.4f} (xM={p['xM_first_peak']:.4f})")


# ============================================================
# 2) internal-field computation for a REAL (rho,c) system
# ============================================================
def compute_full_sphere_intensity_real(rho0, rho1, rho2, c0, c1, c2, delta, xm_ref, res=1000, l_max=None):
    missing = [name for name in ('coeffs', 'J', 'Y') if name not in globals()]
    if missing:
        raise NameError('Define or import these required functions first: ' + ', '.join(missing) + '.')

    a, b, xM = 1.0, 1.0 + delta, float(xm_ref)
    xA = (c0/c1)*xM
    xB = (c0/c2)*xM
    params = {'xA': xA, 'xB': xB, 'yM': b*xM, 'yB': b*xB,
              'rho0': rho0, 'rho1': rho1, 'rho2': rho2,
              'c0': c0, 'c1': c1, 'c2': c2}

    if l_max is None:
        l_max = int(max(15, np.ceil(params['yB'] + 20)))

    limit = 1.15*b
    coords = np.linspace(-limit, limit, int(res))
    X, Y_ = np.meshgrid(coords, coords)
    radius = np.sqrt(X**2 + Y_**2)
    cos_theta = np.divide(Y_, radius, out=np.zeros_like(Y_), where=radius != 0)

    phi = np.zeros_like(radius, dtype=complex)
    core_mask = radius <= a
    shell_mask = (radius > a) & (radius <= b)
    inside_mask = radius <= b

    for ell in range(l_max + 1):
        b_ell, c_ell, d_ell = coeffs(ell, **params)
        factor = (2*ell + 1) * 1j**ell
        legendre = eval_legendre(ell, cos_theta)

        core_radius = xA*radius[core_mask]
        shell_radius = xB*radius[shell_mask]

        phi[core_mask] += factor*b_ell*J(ell, core_radius)*legendre[core_mask]
        shell_term = c_ell*J(ell, shell_radius) + d_ell*Y(ell, shell_radius)
        phi[shell_mask] += factor*shell_term*legendre[shell_mask]

    intensity = np.abs(phi)**2
    intensity[inside_mask] = np.maximum(intensity[inside_mask], 1e-12)

    return {'intensity': intensity,
            'intensity_masked': np.ma.masked_where(~inside_mask, intensity),
            'mask_inside': inside_mask, 'a': a, 'b': b, 'lim': limit,
            'delta': delta, 'xm_ref': xM, 'l_max': l_max}


def save_single_internal_field_image(result, norm, cmap, image_figsize, save_name):
    sizes = scaled_plot_sizes(image_figsize)
    a, b, limit = result['a'], result['b'], result['lim']

    fig, ax = plt.subplots(figsize=image_figsize)
    ax.imshow(result['intensity_masked'], extent=[-limit, limit, -limit, limit],
              origin='lower', cmap=cmap, norm=norm)
    ax.add_artist(plt.Circle((0,0), a, color='black', fill=False, linestyle='--',
                              linewidth=sizes['circle_a_width']))
    ax.add_artist(plt.Circle((0,0), b, color='black', fill=False,
                              linewidth=sizes['circle_b_width']))
    ax.set_aspect('equal')

    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_linewidth(sizes['spine_width'])

    # title shows yM = (1+delta)*xM -- fonte maior, fixa, independente do figsize
    yM_val = (1 + result['delta']) * result['xm_ref']
    ax.set_title(rf"$w_M={yM_val:.4g}$", fontsize=TITLE_FONTSIZE_YM, pad=4.0)

    fig.subplots_adjust(left=0.03, right=0.97, bottom=0.03, top=0.82)
    path = save_name + '.png'
    fig.savefig(path, dpi=600, bbox_inches='tight', pad_inches=0.01)
    plt.show()
    plt.close(fig)
    return path

def save_shared_colorbar(norm, cmap, cbar_figsize, save_name):
    fig = plt.figure(figsize=cbar_figsize, facecolor='white')
    cax = fig.add_axes([0.34, 0.06, 0.3, 0.88])
    mapper = cm.ScalarMappable(norm=norm, cmap=cmap)
    mapper.set_array([])
    cbar = fig.colorbar(mapper, cax=cax)
    cbar.ax.tick_params(labelsize=6.5, length=2.0, width=0.55, pad=1.0)
    for spine in cax.spines.values():
        spine.set_linewidth(0.55)
    path = save_name + '.png'
    fig.savefig(path, dpi=600, bbox_inches=None, pad_inches=0.0, facecolor='white')
    plt.show()
    plt.close(fig)
    return path


# ============================================================
# 3) run for both systems: field near first resonance + field at yM=1
# ============================================================
cmap = make_zero_white_cmap()

for tag, p in systems.items():
    xm_values = (p["xM_first_peak"], p["xM_at_yM1"])

    results = [
        compute_full_sphere_intensity_real(
            p["rho0"], p["rho1"], p["rho2"], p["c0"], p["c1"], p["c2"],
            p["delta"], xm_ref, res=FIELD_RESOLUTION)
        for xm_ref in xm_values
    ]

    all_vals = np.concatenate([
        r['intensity'][r['mask_inside']][
            np.isfinite(r['intensity'][r['mask_inside']]) & (r['intensity'][r['mask_inside']] > 0)]
        for r in results
    ])
    vmin, vmax = np.nanmin(all_vals), np.nanmax(all_vals)
    norm = colors.LogNorm(vmin=vmin, vmax=vmax)

    prefix = os.path.join(OUT_DIR, f"internal_field_{tag}_delta_{number_tag(p['delta'])}")
    image_paths = [
        save_single_internal_field_image(
            r, norm, cmap, IMAGE_FIGSIZE, f"{prefix}_xM{number_tag(r['xm_ref'])}_image")
        for r in results
    ]
    cbar_path = save_shared_colorbar(norm, cmap, CBAR_FIGSIZE, f"{prefix}_shared_cbar")

    print(f"\n{tag} ({p['label']}):")
    print(f"  panel 1 (near first resonance): xM={xm_values[0]:.4f}  (yM={p['yM_first_peak']:.4f})")
    print(f"  panel 2 (yM=1):                 xM={xm_values[1]:.4f}  (yM=1.0000)")
    for path in image_paths:
        print("  ", path)
    print("  ", cbar_path)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as colors

from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import LogFormatterMathtext
from scipy.special import eval_legendre
from scipy.signal import find_peaks


# ============================================================
# Assume já implementados/importados:
#   coeffs, J, Y
#   WT_core_vs_xM, WT_shell_vs_xM
# ============================================================

OUT_DIR = "internal_fields_validation_liu_marston"
os.makedirs(OUT_DIR, exist_ok=True)

LMAX_ENERGY = 10   # apenas para a detecção dos picos


# ============================================================
# FORMATAÇÃO NUMÉRICA: PADRÃO BRASILEIRO
# ============================================================
def fmt_math_br(value, digits=4):
    """
    Formata um número para uso dentro do MathText,
    utilizando vírgula como separador decimal.
    """

    txt = f"{float(value):.{digits}g}"

    # Caso apareça notação científica
    if "e" in txt.lower():
        mantissa, exponent = txt.lower().split("e")

        mantissa = mantissa.replace(".", "{,}")
        exponent = int(exponent)

        return rf"{mantissa}\times 10^{{{exponent}}}"

    return txt.replace(".", "{,}")


def fmt_console_br(value, digits=4):
    """
    Formatação brasileira para valores impressos no console.
    """

    return f"{float(value):.{digits}f}".replace(".", ",")


# ============================================================
# AUXILIARES DE ESTÉTICA
# ============================================================
def make_zero_white_cmap():

    cmap = LinearSegmentedColormap.from_list(
        "white_blue_red_custom",
        [
            (0.0, "white"),
            (0.5, "#355F7C"),
            (1.0, "#990000")
        ],
        N=512
    )

    cmap.set_bad("white")

    return cmap


def eta_metric(alpha, beta):

    return (
        alpha - beta
    ) / (
        alpha + beta
    )


def number_tag(value):

    return (
        f"{float(value):g}"
        .replace("-", "m")
        .replace(".", "p")
    )


def scaled_plot_sizes(figsize):

    size = min(figsize)

    return {
        "title": max(
            8.0,
            2.55 * size
        ),

        "tick_length": max(
            2.4,
            0.75 * size
        ),

        "tick_width": max(
            0.7,
            0.18 * size
        ),

        "spine_width": max(
            0.8,
            0.2 * size
        ),

        "circle_a_width": max(
            0.9,
            0.23 * size
        ),

        "circle_b_width": max(
            1.1,
            0.28 * size
        ),
    }


# ============================================================
# TAMANHO DAS FIGURAS
# ============================================================
IMAGE_FIGSIZE = (3.5, 3.5)

CBAR_FIGSIZE = (
    0.45,
    2.75
)

FIELD_RESOLUTION = 1000

TITLE_FONTSIZE_YM = 15


# ============================================================
# SISTEMAS FÍSICOS
#
# alpha = Z_A / Z_M
# beta  = Z_B / Z_M
# ============================================================
systems = {

    "liu2000": dict(

        label="Liu et al. (2000)",

        # meio externo: epóxi
        rho0=1.18e3,
        c0=2539.52,

        # núcleo: chumbo
        rho1=11.6e3,
        c1=2493.09,

        # casca: borracha de silicone
        rho2=1.3e3,
        c2=22.8709,

        delta=0.4200,
    ),


    "marston2025": dict(

        label="Marston (2025)",

        # meio externo: ar
        rho0=1.22,
        c0=343.0,

        # núcleo: ar
        rho1=1.22,
        c1=343.0,

        # casca: solução de glicerol
        rho2=841.8 * 1.22,
        c2=4.519 * 343.0,

        delta=(1 - 0.7) / 0.7,
    ),
}


# ============================================================
# 1) PRIMEIRA RESSONÂNCIA EM y_M
#
# Convenção baseada no raio externo:
#
# y_M = (1 + delta) x_M
# ============================================================
xM_scan = np.linspace(
    1e-4,
    1.0,
    1000
)


for tag, p in systems.items():

    rho0 = p["rho0"]
    rho1 = p["rho1"]
    rho2 = p["rho2"]

    c0 = p["c0"]
    c1 = p["c1"]
    c2 = p["c2"]

    delta = p["delta"]


    # --------------------------------------------------------
    # Contrastes de impedância
    # --------------------------------------------------------
    alpha = (
        rho1 * c1
    ) / (
        rho0 * c0
    )

    beta = (
        rho2 * c2
    ) / (
        rho0 * c0
    )

    eta = eta_metric(
        alpha,
        beta
    )


    # --------------------------------------------------------
    # Energia no núcleo
    # --------------------------------------------------------
    WT_core = WT_core_vs_xM(
        xM_scan,
        delta,
        rho0,
        rho1,
        rho2,
        c0,
        c1,
        c2,
        lmax_core=LMAX_ENERGY,
        lmax_cap=LMAX_ENERGY
    )


    # --------------------------------------------------------
    # Energia na casca
    # --------------------------------------------------------
    WT_shell = WT_shell_vs_xM(
        xM_scan,
        delta,
        rho0,
        rho1,
        rho2,
        c0,
        c1,
        c2,
        lmax_shell=LMAX_ENERGY,
        lmax_cap=LMAX_ENERGY
    )


    # --------------------------------------------------------
    # Impedâncias
    # --------------------------------------------------------
    Z_A = rho1 * c1
    Z_B = rho2 * c2


    # --------------------------------------------------------
    # Energia interna total
    # --------------------------------------------------------
    W_total = 2.0 * (
        WT_core / Z_A
        + WT_shell / Z_B
    )


    # --------------------------------------------------------
    # Conversão x_M -> y_M
    # --------------------------------------------------------
    yM_scan = (
        1 + delta
    ) * xM_scan


    # --------------------------------------------------------
    # Pontos válidos
    # --------------------------------------------------------
    valid = (
        np.isfinite(W_total)
        & (W_total > 0)
    )

    logW = np.full_like(
        W_total,
        -np.inf
    )

    logW[valid] = np.log10(
        W_total[valid]
    )


    # --------------------------------------------------------
    # Detecção dos picos
    # --------------------------------------------------------
    idx, _ = find_peaks(
        logW,
        prominence=0.75
    )


    if len(idx) > 0:

        yM_first_peak = float(
            yM_scan[idx[0]]
        )

    else:

        yM_first_peak = float(
            yM_scan[
                np.nanargmax(W_total)
            ]
        )


    # --------------------------------------------------------
    # Armazenamento
    # --------------------------------------------------------
    p["alpha"] = alpha
    p["beta"] = beta
    p["eta"] = eta

    p["yM_first_peak"] = (
        yM_first_peak
    )

    p["xM_first_peak"] = (
        yM_first_peak
        / (1 + delta)
    )

    # y_M = 1  ->  x_M = 1/(1 + delta)
    p["xM_at_yM1"] = (
        1.0
        / (1 + delta)
    )


    # --------------------------------------------------------
    # Informações no console
    # --------------------------------------------------------
    print(
        f"{tag}: "
        f"alpha={fmt_console_br(alpha, 4)}  "
        f"beta={fmt_console_br(beta, 4)}  "
        f"delta={fmt_console_br(delta, 4)}  "
        f"eta={fmt_console_br(eta, 4)}  "
        f"| primeira ressonância "
        f"yM={fmt_console_br(yM_first_peak, 4)} "
        f"(xM={fmt_console_br(p['xM_first_peak'], 4)})"
    )


# ============================================================
# 2) CAMPO INTERNO PARA SISTEMA REAL
# ============================================================
def compute_full_sphere_intensity_real(
    rho0,
    rho1,
    rho2,
    c0,
    c1,
    c2,
    delta,
    xm_ref,
    res=1000,
    l_max=None
):

    missing = [
        name
        for name in (
            "coeffs",
            "J",
            "Y"
        )
        if name not in globals()
    ]

    if missing:

        raise NameError(
            "Defina ou importe primeiro as seguintes funções: "
            + ", ".join(missing)
            + "."
        )


    # --------------------------------------------------------
    # Geometria
    # --------------------------------------------------------
    a = 1.0
    b = 1.0 + delta
    xM = float(xm_ref)


    # --------------------------------------------------------
    # Parâmetros de tamanho internos
    # --------------------------------------------------------
    xA = (
        c0 / c1
    ) * xM

    xB = (
        c0 / c2
    ) * xM


    params = {

        "xA": xA,

        "xB": xB,

        "yM": (
            b * xM
        ),

        "yB": (
            b * xB
        ),

        "rho0": rho0,

        "rho1": rho1,

        "rho2": rho2,

        "c0": c0,

        "c1": c1,

        "c2": c2,
    }


    # --------------------------------------------------------
    # Ordem máxima
    # --------------------------------------------------------
    if l_max is None:

        l_max = int(
            max(
                15,
                np.ceil(
                    params["yB"] + 20
                )
            )
        )


    # --------------------------------------------------------
    # Grade espacial
    # --------------------------------------------------------
    limit = 1.15 * b

    coords = np.linspace(
        -limit,
        limit,
        int(res)
    )

    X, Y_ = np.meshgrid(
        coords,
        coords
    )

    radius = np.sqrt(
        X**2
        + Y_**2
    )

    cos_theta = np.divide(
        Y_,
        radius,
        out=np.zeros_like(Y_),
        where=radius != 0
    )


    # --------------------------------------------------------
    # Campo
    # --------------------------------------------------------
    phi = np.zeros_like(
        radius,
        dtype=complex
    )


    # --------------------------------------------------------
    # Regiões
    # --------------------------------------------------------
    core_mask = (
        radius <= a
    )

    shell_mask = (
        (radius > a)
        & (radius <= b)
    )

    inside_mask = (
        radius <= b
    )


    # --------------------------------------------------------
    # Expansão multipolar
    # --------------------------------------------------------
    for ell in range(
        l_max + 1
    ):

        b_ell, c_ell, d_ell = coeffs(
            ell,
            **params
        )

        factor = (
            (2 * ell + 1)
            * 1j**ell
        )

        legendre = eval_legendre(
            ell,
            cos_theta
        )


        # ----------------------------------------------------
        # Núcleo
        # ----------------------------------------------------
        core_radius = (
            xA
            * radius[core_mask]
        )


        # ----------------------------------------------------
        # Casca
        # ----------------------------------------------------
        shell_radius = (
            xB
            * radius[shell_mask]
        )


        phi[core_mask] += (
            factor
            * b_ell
            * J(
                ell,
                core_radius
            )
            * legendre[core_mask]
        )


        shell_term = (
            c_ell
            * J(
                ell,
                shell_radius
            )
            + d_ell
            * Y(
                ell,
                shell_radius
            )
        )


        phi[shell_mask] += (
            factor
            * shell_term
            * legendre[shell_mask]
        )


    # --------------------------------------------------------
    # Intensidade
    # --------------------------------------------------------
    intensity = np.abs(
        phi
    )**2


    intensity[inside_mask] = np.maximum(
        intensity[inside_mask],
        1e-12
    )


    return {

        "intensity":
            intensity,

        "intensity_masked":
            np.ma.masked_where(
                ~inside_mask,
                intensity
            ),

        "mask_inside":
            inside_mask,

        "a":
            a,

        "b":
            b,

        "lim":
            limit,

        "delta":
            delta,

        "xm_ref":
            xM,

        "l_max":
            l_max,
    }


# ============================================================
# SALVAR MAPA DO CAMPO INTERNO
# ============================================================
def save_single_internal_field_image(
    result,
    norm,
    cmap,
    image_figsize,
    save_name
):

    sizes = scaled_plot_sizes(
        image_figsize
    )

    a = result["a"]
    b = result["b"]
    limit = result["lim"]


    fig, ax = plt.subplots(
        figsize=image_figsize
    )


    # --------------------------------------------------------
    # Campo
    # --------------------------------------------------------
    ax.imshow(
        result["intensity_masked"],
        extent=[
            -limit,
            limit,
            -limit,
            limit
        ],
        origin="lower",
        cmap=cmap,
        norm=norm
    )


    # --------------------------------------------------------
    # Interface núcleo-casca
    # --------------------------------------------------------
    ax.add_artist(
        plt.Circle(
            (0, 0),
            a,
            color="black",
            fill=False,
            linestyle="--",
            linewidth=sizes[
                "circle_a_width"
            ]
        )
    )


    # --------------------------------------------------------
    # Interface casca-meio
    # --------------------------------------------------------
    ax.add_artist(
        plt.Circle(
            (0, 0),
            b,
            color="black",
            fill=False,
            linewidth=sizes[
                "circle_b_width"
            ]
        )
    )


    ax.set_aspect(
        "equal"
    )


    # --------------------------------------------------------
    # Sem ticks espaciais
    # --------------------------------------------------------
    ax.set_xticks([])
    ax.set_yticks([])


    # --------------------------------------------------------
    # Moldura
    # --------------------------------------------------------
    for spine in ax.spines.values():

        spine.set_linewidth(
            sizes[
                "spine_width"
            ]
        )


    # ========================================================
    # IDENTIFICAÇÃO DO PARÂMETRO DE TAMANHO
    #
    # y_M = (1 + delta) x_M
    # ========================================================
    yM_val = (
        1
        + result["delta"]
    ) * result["xm_ref"]


    yM_text = fmt_math_br(
        yM_val,
        digits=4
    )


    ax.set_title(
        rf"$y_M={yM_text}$",
        fontsize=TITLE_FONTSIZE_YM,
        pad=4.0
    )


    # --------------------------------------------------------
    # Ajuste da figura
    # --------------------------------------------------------
    fig.subplots_adjust(
        left=0.03,
        right=0.97,
        bottom=0.03,
        top=0.82
    )


    # --------------------------------------------------------
    # Salvamento
    # --------------------------------------------------------
    path = (
        save_name
        + ".png"
    )


    fig.savefig(
        path,
        dpi=600,
        bbox_inches="tight",
        pad_inches=0.01
    )


    plt.show()

    plt.close(
        fig
    )


    return path


# ============================================================
# BARRA DE CORES COMPARTILHADA
# ============================================================
def save_shared_colorbar(
    norm,
    cmap,
    cbar_figsize,
    save_name
):

    fig = plt.figure(
        figsize=cbar_figsize,
        facecolor="white"
    )


    cax = fig.add_axes([
        0.34,
        0.06,
        0.30,
        0.88
    ])


    mapper = cm.ScalarMappable(
        norm=norm,
        cmap=cmap
    )

    mapper.set_array([])


    # --------------------------------------------------------
    # Colorbar logarítmica
    # --------------------------------------------------------
    cbar = fig.colorbar(
        mapper,
        cax=cax,
        format=LogFormatterMathtext()
    )


    cbar.ax.tick_params(
        labelsize=6.5,
        length=2.0,
        width=0.55,
        pad=1.0
    )


    # --------------------------------------------------------
    # Moldura
    # --------------------------------------------------------
    for spine in cax.spines.values():

        spine.set_linewidth(
            0.55
        )


    # --------------------------------------------------------
    # Salvamento
    # --------------------------------------------------------
    path = (
        save_name
        + ".png"
    )


    fig.savefig(
        path,
        dpi=600,
        bbox_inches=None,
        pad_inches=0.0,
        facecolor="white"
    )


    plt.show()

    plt.close(
        fig
    )


    return path


# ============================================================
# 3) GERAR OS CAMPOS PARA OS DOIS SISTEMAS
#
# Painel 1: primeira ressonância
# Painel 2: y_M = 1
# ============================================================
cmap = make_zero_white_cmap()


for tag, p in systems.items():

    xm_values = (
        p["xM_first_peak"],
        p["xM_at_yM1"]
    )


    # --------------------------------------------------------
    # Cálculo dos campos
    # --------------------------------------------------------
    results = [

        compute_full_sphere_intensity_real(

            p["rho0"],

            p["rho1"],

            p["rho2"],

            p["c0"],

            p["c1"],

            p["c2"],

            p["delta"],

            xm_ref,

            res=FIELD_RESOLUTION
        )

        for xm_ref
        in xm_values
    ]


    # --------------------------------------------------------
    # Escala de cores comum aos dois painéis
    # --------------------------------------------------------
    all_vals = np.concatenate([

        r["intensity"][
            r["mask_inside"]
        ][
            np.isfinite(
                r["intensity"][
                    r["mask_inside"]
                ]
            )
            & (
                r["intensity"][
                    r["mask_inside"]
                ]
                > 0
            )
        ]

        for r in results
    ])


    vmin = np.nanmin(
        all_vals
    )

    vmax = np.nanmax(
        all_vals
    )


    norm = colors.LogNorm(
        vmin=vmin,
        vmax=vmax
    )


    # --------------------------------------------------------
    # Prefixo dos arquivos
    # --------------------------------------------------------
    prefix = os.path.join(

        OUT_DIR,

        (
            f"internal_field_"
            f"{tag}_delta_"
            f"{number_tag(p['delta'])}"
        )
    )


    # --------------------------------------------------------
    # Salvar os dois campos
    # --------------------------------------------------------
    image_paths = [

        save_single_internal_field_image(

            r,

            norm,

            cmap,

            IMAGE_FIGSIZE,

            (
                f"{prefix}_"
                f"xM"
                f"{number_tag(r['xm_ref'])}_"
                f"image"
            )
        )

        for r
        in results
    ]


    # --------------------------------------------------------
    # Salvar colorbar compartilhada
    # --------------------------------------------------------
    cbar_path = save_shared_colorbar(

        norm,

        cmap,

        CBAR_FIGSIZE,

        (
            f"{prefix}_"
            f"shared_cbar"
        )
    )


    # ========================================================
    # INFORMAÇÕES NO CONSOLE
    # ========================================================
    print(
        f"\n{tag} ({p['label']}):"
    )


    print(
        "  painel 1 "
        "(próximo à primeira ressonância): "
        f"xM={fmt_console_br(xm_values[0], 4)}  "
        f"(yM={fmt_console_br(p['yM_first_peak'], 4)})"
    )


    print(
        "  painel 2 "
        "(yM=1): "
        f"xM={fmt_console_br(xm_values[1], 4)}  "
        "(yM=1,0000)"
    )


    for path in image_paths:

        print(
            "  ",
            path
        )


    print(
        "  ",
        cbar_path
    )